# QDEGRADE — Predicting Task-Specific Quantization Degradation in Edge LLMs

We ask whether *quantization-induced failure is predictable from the prompt alone*, before the
model runs. If yes, an edge device can route the risky prompts to a full-precision or cloud
model and keep everything else local.

| Stage | What it produces |
|---|---|
| §1–5 | Generations from one model at 4 precisions (fp16, int8, nf4, fp4) |
| **§6 PILOT** | **Go/no-go gate.** Noise floor + degradation rate. Stop here if it fails |
| §7–8 | Degradation labels on the fp16-correct subset |
| §9–10 | Tier-0 features (free text stats) and Tier-1 features (one forward pass) |
| §11–13 | Meta-classifier + the three confound controls that decide acceptance |
| §14 | Router simulation — the headline figure |
| §15–17 | Cross-model transfer, figures, LaTeX tables |

**The label.** Restricted to prompts the fp16 model gets *right*:

| fp16 | quantized | label |
|---|---|---|
| correct | correct | 0 (preserved) |
| correct | wrong | **1 (degradation)** |
| wrong | correct | excluded (recovery) |
| wrong | wrong | excluded |

Excluding the fp16-wrong rows is what makes the label mean *"quantization broke this"*
rather than *"this prompt is hard."*

**Three controls that decide whether reviewers believe you** (§13):
1. **Task-ID-only baseline** — a classifier given nothing but a one-hot task label. Your model must beat it.
2. **Within-task AUC** — computed inside each task separately. This is the number that matters.
3. **Leave-one-task-out** — train on two tasks, test on the third.

Without these, XGBoost learns "GSM8K is harder than SST-2" and you have no paper.

**Run order:** §0 → §1–5 → **§6 (gate)** → §7 onward. Set `SMOKE=True` first and run the
whole notebook end to end in ~2 minutes to prove the analysis pipeline works before you
spend GPU hours.

## 0. Setup and configuration

In [1]:
# Colab: run once. Restart runtime if bitsandbytes complains.
INSTALL = True   # set True on first Colab run
if INSTALL:
    !pip -q install -U transformers accelerate bitsandbytes datasets xgboost pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 MB 3.9 MB/s eta 0:00:00


In [2]:
import os, re, json, math, time, gc, warnings, itertools, hashlib
from collections import Counter
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# ------------------------------------------------------------------ switches
SMOKE       = False     # True  -> synthetic data, no GPU, no downloads. VALIDATE FIRST.
USE_DRIVE   = True     # mount Google Drive for resumable caching
SEED        = 0

# ------------------------------------------------------------------ scope
MODEL_ZOO    = {"qwen1_5b": "Qwen/Qwen2.5-1.5B-Instruct"}
PRIMARY      = "qwen1_5b"
RUN_MODELS   = [PRIMARY]             # single model by design; no cross-model study

REF_PREC     = "fp16"
# Order matters: int8 (LLM.int8) is by far the slowest tier on a T4 -- often 3-5x
# slower than fp16 -- so it runs LAST. If you run out of time or Colab quota, you
# still have both 4-bit tiers, which carry the headline result.
# Four tiers. int8 runs last (slowest). If it is near-lossless that is itself reportable:
# the bit-width at which degradation begins is part of the story.
PRECISIONS   = ["fp16", "nf4", "fp4", "int8"]

# Five tasks. gsm8k_direct uses the SAME problems as gsm8k but forbids reasoning, so
# output length drops from ~512 tokens to ~12 with content held constant. That pair is a
# controlled test separating generation length from reasoning depth -- the confound a
# three-task design cannot address.
TASKS        = ["gsm8k", "gsm8k_direct", "arc_easy", "arc_challenge", "sst2"]
FAMILY       = {"gsm8k": "gsm8k", "gsm8k_direct": "gsm8k", "arc_easy": "arc",
                "arc_challenge": "arc", "sst2": "sst2"}
# Per-task budget. GSM8K carries most of the degradation signal AND has the smallest
# fp16-correct subset, so it gets a larger share.
# ARC (16 new tokens) and SST-2 (6) are nearly free next to GSM8K (512), so they carry
# the larger budget. This is what makes within-task AUC analysable on more than one task.
# SST-2 is capped by its validation split (872 rows).
# Colab Pro budget. More positives is the most reliable way to get a trustworthy AUC:
# narrower CIs and less fold-to-fold variance. SST-2 is capped by its split (872 rows).
N_PER_TASK   = {"gsm8k": 1000, "gsm8k_direct": 1000, "arc_easy": 1500,
                "arc_challenge": 1000, "sst2": 872}
def n_for(task): return (N_PER_TASK[task] if isinstance(N_PER_TASK, dict) else N_PER_TASK)
N_TOTAL      = sum(n_for(t) for t in TASKS)
N_PILOT      = 100                    # per task, for the §6 gate

MAX_NEW = {"gsm8k": 512, "gsm8k_direct": 12, "arc_easy": 16,
           "arc_challenge": 16, "sst2": 6}
BATCH   = {"gsm8k": 8, "gsm8k_direct": 32, "arc_easy": 16,
           "arc_challenge": 16, "sst2": 32}                    # scaled by VRAM below

def autoscale_batch():
    # Colab Pro may hand you an L4 (22 GB) or A100 (40 GB) rather than a T4 (15 GB).
    # Bigger batches are the largest single speedup available for generation.
    try:
        import torch
        if not torch.cuda.is_available(): return "no CUDA -- batch unchanged"
        gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        mult = 1 if gb < 18 else (2 if gb < 32 else 4)
        for k in BATCH: BATCH[k] *= mult
        return f"{torch.cuda.get_device_name(0)} ({gb:.0f} GB) -> batch x{mult}: {BATCH}"
    except Exception as e:
        return f"autoscale skipped: {e}"
MAX_LEN = 640

# ------------------------------------------------------------------ analysis
N_SPLITS, N_REPEATS = 5, 5            # RepeatedStratifiedKFold
HP_SEARCH   = True                    # nested hyperparameter search for the headline arm
HP_ITERS    = 20                      # randomized-search candidates per outer fold
TOPK_KL   = 0                         # 0 = exact KL over full vocab (both models fit on a T4)
USE_SPACY = False                     # optional syntax features; adds install friction
USE_EMB   = False                     # optional MiniLM-embedding baseline arm (§12)
TIER0_ON_ITEM = False                 # True -> Tier-0 features on the benchmark item
                                      # only, excluding the constant instruction (§1b)
SKIP_TIER1 = False                    # True -> skip §10 entirely (saves ~45 min and
                                      # avoids holding two models in VRAM at once)

np.random.seed(SEED)
if not SMOKE: print(autoscale_batch())

# ------------------------------------------------------------------ log control
# bitsandbytes emits a dtype-cast warning on EVERY matmul. Left on, it writes millions
# of lines into the notebook DOM, which exhausts browser memory and takes the machine
# down. This is the single most important line in the cell.
import logging
for _lg in ["bitsandbytes", "bitsandbytes.autograd._functions",
            "bitsandbytes.autograd", "accelerate", "transformers"]:
    logging.getLogger(_lg).setLevel(logging.ERROR)
logging.getLogger("bitsandbytes.autograd._functions").propagate = False
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["BITSANDBYTES_NOWELCOME"] = "1"

# ------------------------------------------------------------------ paths
if USE_DRIVE and not SMOKE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/qdegrade"
else:
    ROOT = "/content/qdegrade" if os.path.isdir("/content") else "./qdegrade"

DIRS = {k: os.path.join(ROOT, k)
        for k in ["runs", "feats", "figs", "tables", "results", "models", "shards"]}
for d in DIRS.values(): os.makedirs(d, exist_ok=True)

SAVE_EVERY = 64          # flush generations to disk every N prompts (crash granularity)

# Cache keys do NOT encode generation settings, so changing MAX_NEW, the prompt template,
# the dtype or the decoding config would silently reuse outputs produced under the old
# settings. GEN_TAG is folded into every cache key: bump it whenever you change any of
# those, and stale files are bypassed instead of silently reused.
GEN_TAG = "g3"           # g1: MAX_NEW 256. g2: 320 + forced fp16. g3: 512 / ARC 16
                         # unchanged in g3 -> existing Qwen generations are all reused

# Weight cache stays on LOCAL disk. Google Drive is a FUSE mount without real mmap
# support, so safetensors loading buffers the entire file into system RAM -- which is
# what exhausts Colab's ~12.7 GB and kills the session. A 3 GB re-download per session
# is far cheaper than a crash.
os.environ["HF_HOME"] = "/content/hf_cache" if os.path.isdir("/content") else "./hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

def ram():
    try:
        import psutil
        v = psutil.virtual_memory()
        return f"RAM {v.used/1e9:.1f}/{v.total/1e9:.1f} GB ({v.percent:.0f}%)"
    except Exception: return "RAM n/a"

# ------------------------------------------------------------------ SMOKE overrides
# Smoke mode exists to validate the PIPELINE, not to produce results, so the expensive
# model-selection arms are switched off and the corpus is shrunk. None of this affects
# a real run.
if SMOKE:
    N_PER_TASK = {k: 200 for k in TASKS}
    N_TOTAL    = sum(n_for(t) for t in TASKS)
    HP_SEARCH, HP_ITERS, N_REPEATS = False, 3, 2

print("SMOKE =", SMOKE, "| output root:", ROOT)

NVIDIA A100-SXM4-40GB (42 GB) -> batch x4: {'gsm8k': 32, 'gsm8k_direct': 128, 'arc_easy': 64, 'arc_challenge': 64, 'sst2': 128}
Mounted at /content/drive
SMOKE = False | output root: /content/drive/MyDrive/qdegrade


### 0b. Crash-safe IO and resume status

Colab free tier drops sessions without warning, and a session killed mid-write leaves a
truncated parquet that would otherwise be silently reloaded as valid data. Every write here
goes to a temp file and is then atomically renamed; every read validates and deletes anything
corrupt so the next run regenerates it.

Run `resume_status()` at the start of any session to see exactly what is already on disk.

**One honest caveat.** Resuming changes batch composition at the boundary, and greedy decoding
on GPU is not bit-deterministic across batch shapes. A run split across two sessions can
therefore differ from an uninterrupted one on a small number of prompts. This is the same
effect the §6 noise floor quantifies, and it is bounded by it, so it does not bias the
comparison between precisions. Do not silently mix a resumed run into a reported result
without checking that the noise floor still dominates it.

In [3]:
import shutil

def save_parquet(df, path):
    tmp = path + ".tmp"
    df.to_parquet(tmp)
    os.replace(tmp, path)              # atomic on the same filesystem

def read_parquet_safe(path):
    try:
        return pd.read_parquet(path)
    except Exception as e:
        print(f"[WARN] corrupt cache removed ({type(e).__name__}): {path}")
        try: os.remove(path)
        except OSError: pass
        return None

def shard_dir(key):
    d = os.path.join(DIRS["shards"], key); os.makedirs(d, exist_ok=True); return d

def load_shards(key):
    out = {}
    d = shard_dir(key)
    for f in sorted(os.listdir(d)):
        if not f.endswith(".parquet"): continue
        p = read_parquet_safe(os.path.join(d, f))
        if p is not None: out.update(dict(zip(p.uid, p.output)))
    return out

def write_shard(key, buf, part):
    save_parquet(pd.DataFrame({"uid": list(buf), "output": list(buf.values())}),
                 os.path.join(shard_dir(key), f"part_{part:05d}.parquet"))

def purge_cache(match, dry_run=True):
    # Remove cached runs/shards whose key contains `match`, e.g. purge_cache("nf4")
    # or purge_cache("gsm8k"). Always inspect with dry_run=True first.
    hits = []
    for d in [DIRS["runs"], DIRS["feats"]]:
        for f in os.listdir(d):
            if match in f and f.endswith(".parquet"): hits.append(os.path.join(d, f))
    for f in os.listdir(DIRS["shards"]):
        if match in f: hits.append(os.path.join(DIRS["shards"], f))
    for h in hits:
        print(("would remove " if dry_run else "removed ") + h)
        if not dry_run:
            shutil.rmtree(h) if os.path.isdir(h) else os.remove(h)
    print(f"{len(hits)} entries matching '{match}'")
    return hits

def resume_status():
    rows = []
    for m in RUN_MODELS:
        for prec in PRECISIONS:
            for task in TASKS:
                for tag in (["a", "b"] if prec == REF_PREC else ["a"]):
                    key = f"{m}__{prec}__{task}__{tag}__{GEN_TAG}"
                    fn = os.path.join(DIRS["runs"], key + ".parquet")
                    n_target = int((PROMPTS.task == task).sum())  # per-task budget
                    if os.path.exists(fn):
                        rows.append(dict(stage="generate", key=key, done=n_target,
                                         total=n_target, state="COMPLETE"))
                    else:
                        d = len(load_shards(key))
                        rows.append(dict(stage="generate", key=key, done=d, total=n_target,
                                         state="partial" if d else "not started"))
        for prec in [p for p in PRECISIONS if p != REF_PREC]:
            key = f"tier1__{m}__{prec}__{GEN_TAG}"
            fn = os.path.join(DIRS["feats"], key + ".parquet")
            d = len(PROMPTS) if os.path.exists(fn) else len(load_shards(key))
            rows.append(dict(stage="tier1", key=key, done=d, total=len(PROMPTS),
                             state="COMPLETE" if os.path.exists(fn)
                                   else ("partial" if d else "not started")))
    S = pd.DataFrame(rows)
    S["pct"] = (100 * S.done / S.total.clip(lower=1)).round(1)
    return S

## 1. Datasets

Three task families chosen so degradation rates differ: multi-step arithmetic, commonsense
multiple choice, and single-label sentiment. That spread is the point of the paper, and it is
also the source of the confound §13 controls for.

In [4]:
def _norm_num(s):
    s = str(s).replace(",", "").replace("$", "").strip().rstrip(".")
    m = re.findall(r"-?\d+\.?\d*", s)
    return m[-1] if m else None

def load_tasks(smoke=False, seed=SEED):
    rng = np.random.RandomState(seed)
    rows = []
    if smoke:
        # synthetic corpus with a genuine latent difficulty signal so the whole
        # downstream pipeline can be validated without a GPU
        vocab = ("the a of and to in that is for with model data system value result set "
                 "compute given where each number total cost rate order find").split()
        hard  = "therefore consequently subsequently nevertheless respectively accordingly".split()
        for t in TASKS:
            base = {"gsm8k": 45, "gsm8k_direct": 45, "arc_easy": 22,
                    "arc_challenge": 28, "sst2": 12}[t]
            for i in range(n_for(t)):
                L = max(6, int(rng.normal(base, base * 0.35)))
                w = list(rng.choice(vocab, L))
                for _ in range(rng.poisson(L / 25)):
                    w.insert(rng.randint(0, len(w)), rng.choice(hard))
                rows.append(dict(task=t, uid=f"{t}-{i:05d}",
                                 prompt=" ".join(w) + " ?", gold="X",
                                 _latent=L))  # used only by the smoke simulator
        return pd.DataFrame(rows)

    from datasets import load_dataset
    # ---- GSM8K: chain-of-thought AND direct-answer over the SAME problems
    d = load_dataset("openai/gsm8k", "main", split="test").shuffle(seed=seed)
    for i, r in enumerate(d.select(range(min(n_for("gsm8k"), len(d))))):
        q, g = r["question"].strip(), _norm_num(r["answer"].split("####")[-1])
        rows.append(dict(task="gsm8k", uid=f"gsm8k-{i:05d}",
            prompt=("Solve the problem. Reason step by step, then give the final answer "
                    "on a new line in exactly this form:\n#### <number>\n\nProblem: " + q),
            gold=g))
        if i < n_for("gsm8k_direct"):
            rows.append(dict(task="gsm8k_direct", uid=f"gsm8k_direct-{i:05d}",
                prompt=("Give only the final numeric answer. Do not explain and do not show "
                        "any working. Reply in exactly this form:\n#### <number>\n\n"
                        "Problem: " + q),
                gold=g))
    # ---- ARC: Easy and Challenge share a format and a parser
    for split_name, tname in [("ARC-Easy", "arc_easy"), ("ARC-Challenge", "arc_challenge")]:
        d = load_dataset("allenai/ai2_arc", split_name, split="test").shuffle(seed=seed)
        kept = 0
        for r in d:
            if kept >= n_for(tname): break
            labs, txts = r["choices"]["label"], r["choices"]["text"]
            if r["answerKey"] not in labs: continue
            opts = "\n".join(f"{a}. {b}" for a, b in zip(labs, txts))
            rows.append(dict(task=tname, uid=f"{tname}-{kept:05d}",
                prompt=("Answer the multiple-choice question with the letter of the correct "
                        "option only.\n\nQuestion: " + r["question"].strip() + "\n" + opts +
                        "\nAnswer:"),
                gold=str(r["answerKey"]).strip()))
            kept += 1
    # ---- SST-2
    d = load_dataset("stanfordnlp/sst2", split="validation").shuffle(seed=seed)
    for i, r in enumerate(d.select(range(min(n_for("sst2"), len(d))))):
        rows.append(dict(task="sst2", uid=f"sst2-{i:05d}",
            prompt=("Classify the sentiment of the sentence as positive or negative. "
                    "Answer with one word.\n\nSentence: " + r["sentence"].strip() +
                    "\nSentiment:"),
            gold="positive" if r["label"] == 1 else "negative"))
    return pd.DataFrame(rows)

_PF = os.path.join(DIRS["runs"], "prompts.parquet")
FORCE_REBUILD_PROMPTS = False
_cached = None if FORCE_REBUILD_PROMPTS else (
    read_parquet_safe(_PF) if os.path.exists(_PF) else None)
if _cached is not None and len(_cached) == N_TOTAL:
    PROMPTS = _cached
    print("[resume] prompts loaded from cache -- no dataset download needed")
else:
    PROMPTS = load_tasks(smoke=SMOKE)
    save_parquet(PROMPTS, _PF)
print(PROMPTS.groupby("task").size().to_string())
print("total prompts:", len(PROMPTS))

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

... (27 outputs left out)

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

task
arc_challenge    1000
arc_easy         1500
gsm8k            1000
gsm8k_direct     1000
sst2              872
total prompts: 5372


### 1b. Inspect and export the prompt set

Two things a reviewer will want and one you need for the manuscript: a look at what the model
actually receives, an example table, and the prompt set released as an artifact.

Note that `chatify()` wraps every prompt in the model chat template before generation, so the
string below is the instruction body, not the literal token sequence. The wrapper is identical
for every prompt, so it does not affect any feature.

`item_text` strips the fixed instruction and keeps only the benchmark item. Set
`TIER0_ON_ITEM = True` in §0 to compute Tier-0 features on the item alone rather than on the
instruction plus item. The instruction is constant within a task, so it inflates cross-task
separability without carrying any per-prompt information. Doing this changes only §9 onward
(CPU, minutes) and requires no regeneration.

In [5]:
def item_text(task, prompt):
    # strip the constant instruction wrapper, keep the benchmark item
    try:
        if FAMILY.get(task) == "gsm8k": return prompt.split("Problem: ", 1)[1].strip()
        if FAMILY.get(task) == "arc":   return prompt.split("Question: ", 1)[1].rsplit("\nAnswer:", 1)[0].strip()
        if task == "sst2":     return prompt.split("Sentence: ", 1)[1].rsplit("\nSentiment:", 1)[0].strip()
    except IndexError: pass
    return prompt

for t in TASKS:
    r = PROMPTS[PROMPTS.task == t].iloc[0]
    print("=" * 78); print(f"TASK: {t}   uid={r.uid}   gold={r.gold!r}")
    print("-" * 78); print(r.prompt)
    print(f"--- max_new_tokens={MAX_NEW[t]} | n_prompts={int((PROMPTS.task==t).sum())}")
print("=" * 78)

# export the prompt set as a releasable artifact
EXPORT = PROMPTS[["uid", "task", "prompt", "gold"]].copy()
EXPORT["item"] = [item_text(t, p) for t, p in zip(EXPORT.task, EXPORT.prompt)]
EXPORT.to_csv(os.path.join(DIRS["results"], "prompt_set.csv"), index=False)
print("exported ->", os.path.join(DIRS["results"], "prompt_set.csv"))

TASK: gsm8k   uid=gsm8k-00000   gold='400'
------------------------------------------------------------------------------
Solve the problem. Reason step by step, then give the final answer on a new line in exactly this form:
#### <number>

Problem: Carmen has $100, Samantha has $25 more than Carmen, and Daisy has $50 more than Samantha. How much do all three girls have combined?
--- max_new_tokens=512 | n_prompts=1000
TASK: gsm8k_direct   uid=gsm8k_direct-00000   gold='400'
------------------------------------------------------------------------------
Give only the final numeric answer. Do not explain and do not show any working. Reply in exactly this form:
#### <number>

Problem: Carmen has $100, Samantha has $25 more than Carmen, and Daisy has $50 more than Samantha. How much do all three girls have combined?
--- max_new_tokens=12 | n_prompts=1000
TASK: arc_easy   uid=arc_easy-00000   gold='D'
------------------------------------------------------------------------------
Answer the m

In [6]:
# paper-ready example table: one short item per task, truncated for a 2-column layout
def _tex_escape(x):
    for a, b in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("$", r"\$"),
                 ("#", r"\#"), ("_", r"\_"), ("{", r"\{"), ("}", r"\}"),
                 ("~", r"\textasciitilde{}"), ("^", r"\textasciicircum{}")]:
        x = x.replace(a, b)
    return x

rows = []
for t in TASKS:
    sub = PROMPTS[PROMPTS.task == t].copy()
    sub["L"] = sub.prompt.str.len()
    r = sub.sort_values("L").iloc[len(sub) // 10]        # a short but typical item
    it = item_text(t, r.prompt).replace("\n", " ")
    rows.append(dict(Task=t, Example=(it[:150] + "...") if len(it) > 150 else it,
                     Gold=str(r.gold), MaxNew=MAX_NEW[t],
                     N=int((PROMPTS.task == t).sum())))
EXAMPLES = pd.DataFrame(rows)
print(EXAMPLES.to_string(index=False))

body = "\\begin{tabular}{p{0.12\\columnwidth}p{0.60\\columnwidth}rr}\n\\hline\n"
body += "Task & Example item & Gold & $N$ \\\\\n\\hline\n"
for _, r in EXAMPLES.iterrows():
    body += (f"{_tex_escape(r.Task)} & {_tex_escape(r.Example)} & "
             f"{_tex_escape(r.Gold)} & {r.N} \\\\\n")
body += "\\hline\n\\end{tabular}"
tex = ("\\begin{table}[t]\n\\centering\n\\caption{Corpus composition with one representative "
       "item per task. Each item is wrapped in a fixed, task-specific instruction before "
       "generation.}\n\\label{tab:examples}\n\\resizebox{\\columnwidth}{!}{%\n"
       + body + "}\n\\end{table}\n")
os.makedirs(DIRS["tables"], exist_ok=True)
open(os.path.join(DIRS["tables"], "T0_examples.tex"), "w").write(tex)
print("\ntable: T0_examples")

         Task                                                                                                                                  Example     Gold  MaxNew    N
        gsm8k A glass of milk is 8 ounces of milk.  John drinks 2 glasses of milk.  If milk has 3 calories per ounce how many calories did he consume?       48     512 1000
 gsm8k_direct A glass of milk is 8 ounces of milk.  John drinks 2 glasses of milk.  If milk has 3 calories per ounce how many calories did he consume?       48      12 1000
     arc_easy                           Which energy source could involve a dam? A. biomass B. solar power C. geothermal energy D. hydroelectric power        D      16 1500
arc_challenge       Which process in the water cycle returns water to the atmosphere? A. condensation B. evaporation C. precipitation D. transpiration        B      16 1000
         sst2                                                                                             too slow , too long and too l

## 2. Answer parsing and scoring

Exact match on the extracted answer. No fuzzy semantic judgement anywhere — a reviewer cannot
argue with a number, but can argue endlessly with "diverges semantically".

In [46]:
def parse_answer(task, text):
    task = FAMILY.get(task, task)
    t = (text or "").strip()
    if task == "gsm8k":
        if "####" in t: return _norm_num(t.split("####")[-1].split("\n")[0])
        return _norm_num(t)
    if task == "arc":
        m = re.search(r"\b([A-E1-4])\b", t)
        return m.group(1) if m else None
    if task == "sst2":
        tl = t.lower()
        pos, neg = tl.find("positive"), tl.find("negative")
        if pos < 0 and neg < 0: return None
        if pos < 0: return "negative"
        if neg < 0: return "positive"
        return "positive" if pos < neg else "negative"
    return None

def is_correct(task, gold, text):
    if SMOKE: return str(text).strip() == "CORRECT"
    fam = FAMILY.get(task, task)
    p = parse_answer(task, text)
    if p is None: return False
    if fam == "gsm8k":
        try: return abs(float(p) - float(gold)) < 1e-4
        except Exception: return False
    return str(p).strip().upper() == str(gold).strip().upper()

In [47]:
def rescore_all():
    fixed = []
    for f in sorted(os.listdir(DIRS["runs"])):
        if not f.endswith(".parquet") or f.startswith(("prompts", "sweep")): continue
        p = os.path.join(DIRS["runs"], f)
        d = read_parquet_safe(p)
        if d is None or "output" not in d.columns: continue
        old = float(d.correct.mean())
        d["correct"] = [is_correct(t, g, o) for t, g, o in zip(d.task, d.gold, d.output)]
        new = float(d.correct.mean())
        if abs(new - old) > 1e-9:
            save_parquet(d, p); fixed.append((f, round(old, 3), round(new, 3)))
    print(f"rescored {len(fixed)} files"); [print(" ", *x) for x in fixed]
rescore_all()

rescored 9 files
  qwen1_5b__fp16__arc_challenge__a__g3.parquet 0.0 0.754
  qwen1_5b__fp16__arc_challenge__a__g3__n100.parquet 0.0 0.76
  qwen1_5b__fp16__arc_challenge__b__g3.parquet 0.0 0.754
  qwen1_5b__fp16__arc_challenge__b__g3__n100.parquet 0.0 0.76
  qwen1_5b__fp4__arc_challenge__a__g3.parquet 0.0 0.674
  qwen1_5b__fp4__arc_challenge__a__g3__n100.parquet 0.0 0.64
  qwen1_5b__int8__arc_challenge__a__g3.parquet 0.0 0.743
  qwen1_5b__nf4__arc_challenge__a__g3.parquet 0.0 0.651
  qwen1_5b__nf4__arc_challenge__a__g3__n100.parquet 0.0 0.63


## 3. Model loading

`fp16` is the reference. `int8`, `nf4` and `fp4` all come from `bitsandbytes`, so there is no
extra dependency and no separate quantized checkpoint to download. `fp4` is the harshest tier available from `bitsandbytes` here, and is your insurance policy if
`nf4` turns out too clean to produce positives.

Note that `bnb_4bit_use_double_quant` is **not** a harshness dial: double quantization only
compresses the quantization constants, saving roughly 0.4 bits per parameter with a negligible
accuracy effect. If both 4-bit tiers come back clean, the real next step is a prequantized
3-bit GPTQ or AWQ checkpoint. Report every tier you ran either way; choosing which tier to
headline is legitimate, engineering a tier to manufacture positives is not.

In [8]:
def load_model(model_key, precision):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    name = MODEL_ZOO[model_key]
    tok = AutoTokenizer.from_pretrained(name, padding_side="left")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    # float16 base dtype for EVERY tier: the T4 has no bf16 support, and letting a
    # quantized tier default to the config's bf16 both floods the log with cast
    # warnings and makes the comparison against the fp16 reference inconsistent.
    kw = dict(device_map={"": 0}, low_cpu_mem_usage=True, torch_dtype=torch.float16)
    if precision == "fp16":
        pass
    elif precision == "int8":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    elif precision == "nf4":
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    elif precision == "fp4":
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="fp4",
            bnb_4bit_use_double_quant=False, bnb_4bit_compute_dtype=torch.float16)
    else:
        raise ValueError(precision)
    import logging as _l
    _l.getLogger("bitsandbytes.autograd._functions").setLevel(_l.ERROR)
    try:
        from transformers.utils import logging as _tl; _tl.set_verbosity_error()
    except Exception: pass
    m = AutoModelForCausalLM.from_pretrained(name, **kw).eval()
    return m, tok

def free():
    # Callers must `del` their own references FIRST -- deleting a function-local copy
    # of a reference does not release the caller's. This only sweeps up afterwards.
    for _ in range(3): gc.collect()
    try:
        import torch
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    except Exception: pass

def chatify(tok, prompt):
    try:
        return tok.apply_chat_template([{"role": "user", "content": prompt}],
                                       tokenize=False, add_generation_prompt=True)
    except Exception:
        return prompt

## 4. Generation

Greedy decoding throughout. Note that greedy is **not** bit-deterministic on GPU: kernel
reduction order changes with batch composition, so two fp16 runs can disagree. That is exactly
what §6 measures, and `shuffle=True` in the second fp16 pass deliberately perturbs batching to
expose it.

In [9]:
def generate_run(model_key, precision, task, run_tag="a", shuffle=False, n=None,
                 seed=SEED, model=None, tok=None):
    key = (f"{model_key}__{precision}__{task}__{run_tag}__{GEN_TAG}"
           f"{'' if n is None else '__n' + str(n)}")
    fn = os.path.join(DIRS["runs"], key + ".parquet")

    sub = PROMPTS[PROMPTS.task == task].copy()
    if n is not None: sub = sub.head(n)

    if os.path.exists(fn):
        d = read_parquet_safe(fn)
        # Only reuse a completed file if it covers the CURRENT budget. Raising
        # N_PER_TASK would otherwise be silently ignored, since the cache key does
        # not encode the prompt count.
        if d is not None and len(d) >= len(sub):
            return d
        if d is not None:
            print(f"  [grow] {key}: cached {len(d)} < target {len(sub)}, extending")

    done = load_shards(key)                       # <-- prompt-level resume
    todo = sub[~sub.uid.isin(done)]
    if len(done): print(f"  [resume] {key}: {len(done)} cached, {len(todo)} remaining")

    if len(todo):
        part = len([f for f in os.listdir(shard_dir(key)) if f.endswith(".parquet")])
        if SMOKE:
            g = _smoke_generate(todo, precision, run_tag, seed)
            buf = dict(zip(g.uid, g.output)); write_shard(key, buf, part); done.update(buf)
        else:
            import torch
            owns = model is None
            if owns:                                   # load only if caller did not
                model, tok = load_model(model_key, precision)
            texts = [chatify(tok, p) for p in todo.prompt]; uids = list(todo.uid)
            order = np.argsort([len(t) for t in texts])      # length-sorted batching
            if shuffle:
                order = np.random.RandomState(seed + 1).permutation(len(texts))
            bs = BATCH[task] * (2 if shuffle else 1)
            buf, t0 = {}, time.time()
            for i in range(0, len(order), bs):
                idx = order[i:i + bs]
                enc = tok([texts[j] for j in idx], return_tensors="pt", padding=True,
                          truncation=True, max_length=MAX_LEN).to("cuda")
                with torch.no_grad():
                    g = model.generate(**enc, max_new_tokens=MAX_NEW[task], do_sample=False,
                                       temperature=None, top_p=None,
                                       pad_token_id=tok.pad_token_id)
                for k, j in enumerate(idx):
                    buf[uids[j]] = tok.decode(g[k][enc["input_ids"].shape[1]:],
                                              skip_special_tokens=True)
                # flush to Drive every SAVE_EVERY prompts: a session kill costs at
                # most SAVE_EVERY prompts of work, not the whole task
                if len(buf) >= SAVE_EVERY or i + bs >= len(order):
                    write_shard(key, buf, part); done.update(buf); buf = {}; part += 1
                    print(f"  {key} {len(done)}/{len(sub)} "
                          f"{time.time()-t0:.0f}s | {ram()}", flush=True)
                del enc, g
            if owns:
                del model, tok
                free()
            del texts, uids
    sub["output"] = [done[u] for u in sub.uid]
    sub["correct"] = [is_correct(task, g, o) for g, o in zip(sub.gold, sub.output)]
    sub["precision"], sub["model"], sub["run_tag"] = precision, model_key, run_tag
    save_parquet(sub, fn)                 # complete: shards for this key are now redundant
    return sub

### 4b. Smoke simulator

Fabricates outputs whose correctness depends on a latent difficulty that the Tier-0 features
can partly recover, plus per-precision damage and a small nondeterminism term. This exists so
the entire analysis pipeline (§7–17) can be validated in minutes without a GPU. It is **not**
part of the paper.

In [10]:
_PREC_DAMAGE = {"fp16": 0.00, "int8": 0.04, "nf4": 0.11, "fp4": 0.20}

FLIP_RATE = 0.02   # simulated run-to-run nondeterminism (the fp16 noise floor)

def _smoke_generate(sub, precision, run_tag, seed):
    lat = sub["_latent"].values.astype(float)
    lat = (lat - lat.mean()) / (lat.std() + 1e-9)
    base_acc = 1 / (1 + np.exp(0.8 * lat - 0.6))                   # long prompts are harder
    dmg = _PREC_DAMAGE[precision] * (1 / (1 + np.exp(-0.9 * lat)))  # damage tracks difficulty
    p_ok = np.clip(base_acc - dmg, 0.01, 0.99)
    # per-prompt draw is deterministic in the uid, so runs a and b agree except for flips
    u = np.array([int(hashlib.md5(f"{x}{seed}".encode()).hexdigest()[:8], 16) / 2**32
                  for x in sub.uid])
    ok = u < p_ok
    # flip mask is keyed on the uid, not the row position, so a resumed partial run
    # reproduces exactly what an uninterrupted run would have produced
    flip = np.array([int(hashlib.md5(f"{x}{precision}{run_tag}{seed}".encode())
                         .hexdigest()[8:16], 16) / 2**32 for x in sub.uid]) < FLIP_RATE
    ok = np.where(flip, ~ok, ok)
    return sub.assign(output=["CORRECT" if o else "WRONG" for o in ok])

## 5. Full generation sweep

Resumable: every (model, precision, task, run_tag) is cached as its own parquet. If Colab
kills the session, re-run this cell and it picks up where it stopped.

`fp16` is generated **twice** (`a` and `b`) with different batching. Run `b` is not a duplicate
— it is the measurement instrument for the noise floor.

In [48]:
KEEP = ["uid", "task", "gold", "correct", "precision", "model", "run_tag"]

def sweep(model_key, n=None, precisions=None, tag=""):
    # One model load per PRECISION, with all tasks generated inside that load.
    # The old loop reloaded weights 12 times and never released them, which is the
    # other half of the RAM problem.
    precisions = precisions or PRECISIONS
    frames = []
    for prec in precisions:
        need = any(not os.path.exists(os.path.join(
                       DIRS["runs"],
                       f"{model_key}__{prec}__{t}__a__{GEN_TAG}"
                       f"{'' if n is None else '__n' + str(n)}.parquet"))
                   for t in TASKS)
        model = tok = None
        try:
            if need and not SMOKE:
                model, tok = load_model(model_key, prec)
                print(f"[load] {prec} | {ram()}", flush=True)
            for task in TASKS:
                frames.append(generate_run(model_key, prec, task, "a", n=n,
                                           model=model, tok=tok)[KEEP])
                if prec == REF_PREC:      # second reference pass -> noise floor
                    frames.append(generate_run(model_key, prec, task, "b", shuffle=True,
                                               n=n, model=model, tok=tok)[KEEP])
        except Exception as e:
            print(f"[WARN] {model_key}/{prec} failed -> skipped: "
                  f"{type(e).__name__}: {e}")
        finally:
            del model, tok
            free()
            print(f"[unload] {prec} | {ram()}", flush=True)
    df = pd.concat(frames, ignore_index=True)
    save_parquet(df, os.path.join(DIRS["runs"], f"sweep_{model_key}{tag}.parquet"))
    return df

PILOT = sweep(PRIMARY, n=N_PILOT, precisions=[REF_PREC, "nf4", "fp4"], tag="_pilot")
print(PILOT.groupby(["task", "precision", "run_tag"]).correct.mean().round(3).to_string())

[unload] fp16 | RAM 16.9/89.6 GB (20%)
[unload] nf4 | RAM 16.9/89.6 GB (20%)
[unload] fp4 | RAM 16.9/89.6 GB (20%)
task           precision  run_tag
arc_challenge  fp16       a          0.76
                          b          0.76
               fp4        a          0.64
               nf4        a          0.63
arc_easy       fp16       a          0.91
                          b          0.91
               fp4        a          0.87
               nf4        a          0.88
gsm8k          fp16       a          0.62
                          b          0.62
               fp4        a          0.40
               nf4        a          0.53
gsm8k_direct   fp16       a          0.10
                          b          0.10
               fp4        a          0.06
               nf4        a          0.07
sst2           fp16       a          0.92
                          b          0.92
               fp4        a          0.90
               nf4        a          0.95


## 6. GO / NO-GO GATE

Run this on day one before anything else. Three conditions must hold:

1. **fp16 accuracy is non-trivial** (>0.25) — otherwise there is no fp16-correct subset to work with.
2. **Degradation rate ≥ 5%** on the fp16-correct subset — otherwise you will not have enough positives.
3. **Degradation ≥ 2× the fp16-vs-fp16 noise floor** — otherwise you are modelling GPU nondeterminism.

If `nf4` fails but `fp4` passes, make `fp4` your headline precision and keep `nf4` as the
"benign compression" contrast. If both fail, the design is dead — switch models or go harsher,
do not proceed.

In [49]:
def gate(df, ref=REF_PREC):
    A = df[(df.precision == ref) & (df.run_tag == "a")].set_index("uid")
    B = df[(df.precision == ref) & (df.run_tag == "b")].set_index("uid")
    rep = []
    for task in TASKS:
        a = A[A.task == task]; b = B[B.task == task]
        common = a.index.intersection(b.index)
        a, b = a.loc[common], b.loc[common]
        acc = a.correct.mean()
        okA = a[a.correct]
        noise = (~b.loc[okA.index].correct).mean() if len(okA) else np.nan   # fp16 vs fp16
        row = dict(task=task, n=len(a), fp16_acc=acc, n_fp16_correct=len(okA),
                   noise_floor=noise)
        for prec in [p for p in df.precision.unique() if p != ref]:
            q = df[(df.precision == prec) & (df.run_tag == "a") & (df.task == task)] \
                    .set_index("uid")
            idx = okA.index.intersection(q.index)
            row[f"degrade_{prec}"] = (~q.loc[idx].correct).mean() if len(idx) else np.nan
        rep.append(row)
    return pd.DataFrame(rep)

G = gate(PILOT)
print(G.round(3).to_string(index=False)); print()

for prec in [p for p in PILOT.precision.unique() if p != REF_PREC]:
    # per-task projection, weighted by that task's own fp16-correct subset and budget.
    # An unweighted mean of rates overestimates whenever the high-degradation task
    # also has the smallest correct subset -- which is exactly the GSM8K case.
    per = {r.task: r.n_fp16_correct * (n_for(r.task) / max(N_PILOT, 1))
                   * (0 if not np.isfinite(r[f"degrade_{prec}"])
                      else r[f"degrade_{prec}"]) for _, r in G.iterrows()}
    proj = sum(per.values())
    tot_correct = sum(r.n_fp16_correct * (n_for(r.task) / max(N_PILOT, 1))
                      for _, r in G.iterrows())
    d, nz = proj / max(tot_correct, 1), G.noise_floor.mean()
    print("    projected positives per task: " +
          ", ".join(f"{k}={v:.0f}" for k, v in per.items()))
    c1, c2, c3 = G.fp16_acc.mean() > .25, d >= .05, d >= 2 * max(nz, 1e-9)
    c4 = proj >= 100
    print(f"    tasks with zero positives: "
          f"{[k for k, v in per.items() if v < 1] or 'none'}")
    print(f"[{prec}] pooled degradation={d:.3f}  noise_floor={nz:.3f}  "
          f"projected positives at full scale={proj:.0f}")
    print(f"    acc>0.25 {'PASS' if c1 else 'FAIL'} | "
          f"degrade>=5% {'PASS' if c2 else 'FAIL'} | "
          f"degrade>=2x noise {'PASS' if c3 else 'FAIL'} | "
          f">=100 positives {'PASS' if c4 else 'FAIL'} | "
          f"==> {'GO' if (c1 and c2 and c3 and c4) else 'NO-GO'}")
G.to_csv(os.path.join(DIRS["results"], "gate_pilot.csv"), index=False)

         task   n  fp16_acc  n_fp16_correct  noise_floor  degrade_nf4  degrade_fp4
        gsm8k 100      0.62              62        0.016        0.290        0.468
 gsm8k_direct 100      0.10              10        0.000        0.400        0.600
     arc_easy 100      0.91              91        0.000        0.044        0.088
arc_challenge 100      0.76              76        0.000        0.171        0.184
         sst2 100      0.92              92        0.000        0.000        0.043

    projected positives per task: gsm8k=180, gsm8k_direct=40, arc_easy=60, arc_challenge=130, sst2=0
    tasks with zero positives: ['sst2']
[nf4] pooled degradation=0.112  noise_floor=0.003  projected positives at full scale=410
    acc>0.25 PASS | degrade>=5% PASS | degrade>=2x noise PASS | >=100 positives PASS | ==> GO
    projected positives per task: gsm8k=290, gsm8k_direct=60, arc_easy=120, arc_challenge=140, sst2=35
    tasks with zero positives: none
[fp4] pooled degradation=0.177  noise_

### 6b. Generation diagnostics — run this if fp16 accuracy looks low

A low fp16 accuracy shrinks the usable subset **and** may bias it toward short, easy prompts,
which would distort every downstream result. Two failure modes look identical in the accuracy
number but need different fixes:

- **Truncation** — the chain of thought hits `MAX_NEW` before reaching the answer. Fix by raising `MAX_NEW["gsm8k"]`.
- **Format non-compliance** — the model reasons correctly but never emits `#### <number>`, so the parser scores it wrong. Fix by loosening the parser or the prompt, not by raising token budget.

Anything above roughly 5% unparseable means you are measuring your parser, not the model.

In [50]:
def diagnose(model_key=PRIMARY, precision=REF_PREC, n=N_PILOT, show=4):
    rows = []
    for task in TASKS:
        d = generate_run(model_key, precision, task, "a", n=n)
        out = d.output.fillna("")
        unparsed = [parse_answer(task, o) is None for o in out]
        # crude truncation proxy: no terminal marker and near the token ceiling
        long_out = out.str.len() > (MAX_NEW[task] * 3.2)
        rows.append(dict(task=task, n=len(d), acc=d.correct.mean(),
                         unparsed_rate=float(np.mean(unparsed)),
                         maybe_truncated=float(long_out.mean()),
                         mean_out_chars=float(out.str.len().mean())))
    R = pd.DataFrame(rows)
    print(R.round(3).to_string(index=False))
    for task in TASKS:
        d = generate_run(model_key, precision, task, "a", n=n)
        bad = d[[parse_answer(task, o) is None for o in d.output.fillna("")]]
        if len(bad):
            print(f"\n--- {task}: {len(bad)} unparseable, showing {min(show,len(bad))} ---")
            for _, r in bad.head(show).iterrows():
                print("  gold:", r.gold, "| output tail:", repr(str(r.output)[-160:]))
    return R

DIAG = diagnose()
DIAG.to_csv(os.path.join(DIRS["results"], "diagnostics.csv"), index=False)

         task   n  acc  unparsed_rate  maybe_truncated  mean_out_chars
        gsm8k 100 0.62           0.00             0.02          959.87
 gsm8k_direct 100 0.10           0.00             0.01            3.93
     arc_easy 100 0.91           0.00             0.37           39.20
arc_challenge 100 0.76           0.03             0.62           53.78
         sst2 100 0.92           0.00             0.00            8.00

--- arc_challenge: 3 unparseable, showing 3 ---
  gold: C | output tail: 'To find the average speed of the turtle, we need to use the formula for'
  gold: C | output tail: "To determine the net force acting on the car, we can use Newton's Second"
  gold: C | output tail: 'The best example of an activity that has a positive effect on the habitat of other'


## 7. Full sweep (only after the gate passes)

In [51]:
RUNS = {m: sweep(m) for m in RUN_MODELS}
DF = pd.concat(RUNS.values(), ignore_index=True)
print(DF.groupby(["model", "precision", "task"]).correct.mean().round(3).unstack().to_string())

[unload] fp16 | RAM 17.0/89.6 GB (20%)
[unload] nf4 | RAM 17.0/89.6 GB (20%)
[unload] fp4 | RAM 17.0/89.6 GB (20%)
[unload] int8 | RAM 17.0/89.6 GB (20%)
task                arc_challenge  arc_easy  gsm8k  gsm8k_direct   sst2
model    precision                                                     
qwen1_5b fp16               0.754     0.891  0.670         0.073  0.911
         fp4                0.674     0.847  0.451         0.047  0.867
         int8               0.743     0.889  0.668         0.077  0.907
         nf4                0.651     0.848  0.601         0.065  0.906


### 7b. Truncation diagnostic

Quantized models tend to ramble, so they hit the token budget more often, and a truncated
chain scores as wrong. That is real degradation under a fixed edge budget, but it is a
different mechanism from a wrong answer and a reviewer will ask you to separate them.

Read `deg_from_truncation`: under ~0.15 and one sentence in the paper covers it; above ~0.40
and the honest framing is that quantization degrades output *length discipline*, which is a
legitimate finding but a different one.

In [52]:
def truncation_report(task="gsm8k", tag="a"):
    if SMOKE: return pd.DataFrame([{"note": "not meaningful in SMOKE mode"}])
    from transformers import AutoTokenizer
    _tk = AutoTokenizer.from_pretrained(MODEL_ZOO[PRIMARY])

    def load(prec):
        fn = os.path.join(DIRS["runs"],
                          f"{PRIMARY}__{prec}__{task}__{tag}__{GEN_TAG}.parquet")
        if not os.path.exists(fn): return None
        d = read_parquet_safe(fn)
        n = d.output.map(lambda t: len(_tk(t, add_special_tokens=False)["input_ids"]))
        d["trunc"] = (n >= MAX_NEW[task] - 2)
        return d.set_index("uid")

    ref = load(REF_PREC)
    if ref is None: return pd.DataFrame([{"note": "reference run not found"}])
    ok, rows = ref[ref.correct], []
    for prec in [p for p in PRECISIONS if p != REF_PREC]:
        q = load(prec)
        if q is None: continue
        idx = ok.index.intersection(q.index)
        deg = idx[~q.loc[idx, "correct"].values]
        rows.append(dict(precision=prec, trunc_rate=q.trunc.mean(),
                         trunc_rate_fp16=ref.trunc.mean(), n_degraded=len(deg),
                         deg_from_truncation=q.loc[deg, "trunc"].mean() if len(deg) else np.nan,
                         deg_from_wrong_answer=(1 - q.loc[deg, "trunc"].mean())
                                               if len(deg) else np.nan))
    return pd.DataFrame(rows)

TRUNC = truncation_report()
print(TRUNC.round(3).to_string(index=False))
if not SMOKE and "trunc_rate" in TRUNC:
    TRUNC.to_csv(os.path.join(DIRS["results"], "truncation.csv"), index=False)

precision  trunc_rate  trunc_rate_fp16  n_degraded  deg_from_truncation  deg_from_wrong_answer
      nf4       0.020            0.031         154                0.052                  0.948
      fp4       0.055            0.031         303                0.086                  0.914
     int8       0.039            0.031          75                0.120                  0.880


## 8. Degradation labels

One row per (model, precision, prompt), restricted to the fp16-correct subset.
`y = 1` iff the quantized model gets wrong what fp16 got right.

In [53]:
def build_labels(df):
    out = []
    for model_key, g in df.groupby("model"):
        ref = g[(g.precision == REF_PREC) & (g.run_tag == "a")].set_index("uid")
        refb = g[(g.precision == REF_PREC) & (g.run_tag == "b")].set_index("uid")
        ok = ref[ref.correct]
        for prec in [p for p in PRECISIONS if p != REF_PREC]:
            q = g[(g.precision == prec) & (g.run_tag == "a")].set_index("uid")
            idx = ok.index.intersection(q.index)
            out.append(pd.DataFrame(dict(
                model=model_key, precision=prec, uid=idx,
                task=ok.loc[idx, "task"].values,
                y=(~q.loc[idx, "correct"]).astype(int).values)))
        # fp16 self-disagreement, carried as the noise-floor reference arm
        idx = ok.index.intersection(refb.index)
        out.append(pd.DataFrame(dict(
            model=model_key, precision="fp16_selfrep", uid=idx,
            task=ok.loc[idx, "task"].values,
            y=(~refb.loc[idx, "correct"]).astype(int).values)))
    return pd.concat(out, ignore_index=True)

LAB = build_labels(DF)
save_parquet(LAB, os.path.join(DIRS["feats"], "labels.parquet"))
print(LAB.groupby(["model", "precision"]).y.agg(["mean", "sum", "size"]).round(3).to_string())

                        mean  sum  size
model    precision                     
qwen1_5b fp16_selfrep  0.002    9  3625
         fp4           0.172  625  3625
         int8          0.036  132  3625
         nf4           0.119  430  3625


### 8b. Flip decomposition: why aggregate accuracy hides the problem

Aggregate accuracy nets degradations against recoveries, so a scheme can churn twice as many
individual predictions while reporting the same headline number. That is the gap this paper
exists to close, and this table is the cleanest way to show it.

`churn_ratio` = total flipped predictions divided by the absolute net accuracy change. A value
near 1 means the headline number tells the whole story. A large value means it hides most of
what the quantization actually did.

In [54]:
def flip_decomposition(df):
    rows = []
    for (m, task), g in df[df.run_tag == "a"].groupby(["model", "task"]):
        ref = g[g.precision == REF_PREC].set_index("uid")
        for prec in [p for p in PRECISIONS if p != REF_PREC]:
            q = g[g.precision == prec].set_index("uid")
            idx = ref.index.intersection(q.index)
            if not len(idx): continue
            rc, qc = ref.loc[idx, "correct"].values, q.loc[idx, "correct"].values
            deg, rec = int((rc & ~qc).sum()), int((~rc & qc).sum())
            net = qc.mean() - rc.mean()
            rows.append(dict(model=m, task=task, precision=prec, n=len(idx),
                             fp16_acc=rc.mean(), quant_acc=qc.mean(), net_delta=net,
                             degraded=deg, recovered=rec,
                             degrade_rate=deg / max(rc.sum(), 1),
                             churn=(deg + rec) / len(idx),
                             # undefined when the net change is ~0 -- which is precisely
                             # the most interesting case, so flag it rather than printing
                             # a meaningless 2.7e7
                             churn_ratio=(np.nan if abs(net) < 0.005
                                          else ((deg + rec) / len(idx)) / abs(net)),
                             net_is_zero=bool(abs(net) < 0.005)))
    return pd.DataFrame(rows)

FLIP = flip_decomposition(DF)
print(FLIP.round(3).to_string(index=False))
FLIP.to_csv(os.path.join(DIRS["results"], "flip_decomposition.csv"), index=False)

zero = FLIP[FLIP.net_is_zero]
if len(zero):
    print("\nTiers that look LOSSLESS on aggregate accuracy but are not:")
    for _, r in zero.iterrows():
        print(f"  {r.task}/{r.precision}: net {r.net_delta*100:+.2f} pts, yet "
              f"{r.degraded} degraded and {r.recovered} recovered "
              f"({r.churn*100:.1f}% of predictions changed).")
fin = FLIP[FLIP.churn_ratio.notna()]
if len(fin):
    w = fin.loc[fin.churn_ratio.idxmax()]
    print(f"\nLargest finite ratio: {w.task}/{w.precision} moves {w.churn*100:.1f}% of "
          f"predictions for {w.net_delta*100:+.1f} pts net ({w.churn_ratio:.1f}x).")

   model          task precision    n  fp16_acc  quant_acc  net_delta  degraded  recovered  degrade_rate  churn  churn_ratio  net_is_zero
qwen1_5b arc_challenge       nf4 1000     0.754      0.651     -0.103       123         20         0.163  0.143        1.388        False
qwen1_5b arc_challenge       fp4 1000     0.754      0.674     -0.080       111         31         0.147  0.142        1.775        False
qwen1_5b arc_challenge      int8 1000     0.754      0.743     -0.011        19          8         0.025  0.027        2.455        False
qwen1_5b      arc_easy       nf4 1500     0.889      0.848     -0.041        99         37         0.074  0.091        2.194        False
qwen1_5b      arc_easy       fp4 1500     0.889      0.847     -0.042       114         51         0.085  0.110        2.619        False
qwen1_5b      arc_easy      int8 1500     0.889      0.889      0.000        20         20         0.015  0.027          NaN         True
qwen1_5b         gsm8k       nf4 1

### 8c. Length versus content: the controlled contrast

`gsm8k` and `gsm8k_direct` contain the **same problems**. Only the instruction differs, so
generated length collapses from ~512 tokens to ~12 while reasoning demand is unchanged.

- Similar degradation in both -> damage is driven by task **content**.
- Much lower degradation in `gsm8k_direct` -> damage **accumulates over generated tokens**,
  and the headline result is partly about generation length.

Either answer is publishable. What is not defensible is a design that cannot distinguish
them, which is exactly what a reviewer will raise. `arc_easy` vs `arc_challenge` gives the
same contrast for difficulty at a fixed output length.

In [55]:
PAIRS = [("gsm8k", "gsm8k_direct", "generation length (same items)"),
         ("arc_easy", "arc_challenge", "item difficulty (same format)")]
rows = []
for a, b, axis in PAIRS:
    if not {a, b} <= set(LAB.task.unique()): continue
    for prec in [p for p in PRECISIONS if p != REF_PREC]:
        ga = LAB[(LAB.task == a) & (LAB.precision == prec)]
        gb = LAB[(LAB.task == b) & (LAB.precision == prec)]
        if not (len(ga) and len(gb)): continue
        rows.append(dict(axis=axis, precision=prec,
                         task_a=a, deg_a=ga.y.mean(), n_a=len(ga),
                         task_b=b, deg_b=gb.y.mean(), n_b=len(gb),
                         ratio=ga.y.mean() / max(gb.y.mean(), 1e-9)))
CONTRAST = pd.DataFrame(rows)
print(CONTRAST.round(3).to_string(index=False) if len(CONTRAST) else "no contrast pairs")

sub = CONTRAST[CONTRAST.task_a == "gsm8k"] if len(CONTRAST) else CONTRAST
if len(sub):
    print(f"\nCoT degrades {sub.ratio.mean():.1f}x as often as direct-answer on the same "
          f"problems.")
    print("  ~1x  -> content-driven damage; the length confound is ruled out.")
    print("  >>1x -> damage accumulates over generated tokens; state this explicitly.")

                          axis precision   task_a  deg_a  n_a        task_b  deg_b  n_b  ratio
generation length (same items)       nf4    gsm8k  0.230  670  gsm8k_direct  0.493   73  0.466
generation length (same items)       fp4    gsm8k  0.452  670  gsm8k_direct  0.699   73  0.647
generation length (same items)      int8    gsm8k  0.112  670  gsm8k_direct  0.151   73  0.743
 item difficulty (same format)       nf4 arc_easy  0.074 1334 arc_challenge  0.163  754  0.455
 item difficulty (same format)       fp4 arc_easy  0.085 1334 arc_challenge  0.147  754  0.580
 item difficulty (same format)      int8 arc_easy  0.015 1334 arc_challenge  0.025  754  0.595

CoT degrades 0.6x as often as direct-answer on the same problems.
  ~1x  -> content-driven damage; the length confound is ruled out.
  >>1x -> damage accumulates over generated tokens; state this explicitly.


## 9. Tier-0 features — negligible overhead

Pure text statistics, computable on CPU before any inference happens. Extraction cost is
measured in §17, not assumed to be zero. This is the tier that
matters for the deployment story: if it works, routing costs nothing.

spaCy parse-depth features are optional (`USE_SPACY`), with comma/clause-marker counts as the
dependency-free substitute so the notebook has no extra download.

In [56]:
STOP = set(("the a an and or but if of to in on at for with as is are was were be been by "
            "that this it its from not no so than then there their they he she we you i").split())
CLAUSE = ("if when while because although unless whereas after before since therefore "
          "however moreover thus hence").split()

_TOK0 = None
def _tok0():
    # CPU-only fast tokenizer, loaded once. No model weights, no inference.
    global _TOK0
    if _TOK0 is None and not SMOKE:
        from transformers import AutoTokenizer
        _TOK0 = AutoTokenizer.from_pretrained(MODEL_ZOO[PRIMARY], use_fast=True)
    return _TOK0

def tier0(text):
    t = text
    words = re.findall(r"[A-Za-z']+", t.lower())
    sents = [s for s in re.split(r"[.!?\n]+", t) if s.strip()]
    n_w = max(len(words), 1)
    cnt = Counter(words)
    p = np.array(list(cnt.values()), dtype=float); p /= p.sum()
    chars = Counter(t)
    pc = np.array(list(chars.values()), dtype=float); pc /= pc.sum()
    nums   = re.findall(r"\d+\.?\d*", t)
    fnums  = []
    for x in nums:
        try: fnums.append(float(x))
        except ValueError: pass
    ops    = re.findall(r"[+\-*/=%]", t)
    caps   = re.findall(r"\b[A-Z][a-z]+", t)
    longw  = [w for w in words if len(w) > 6]
    syl    = sum(max(1, len(re.findall(r"[aeiouy]+", w))) for w in words)
    f = dict(
        t0_n_chars      = len(t),
        t0_n_words      = n_w,
        t0_n_sents      = len(sents),
        t0_avg_word_len = float(np.mean([len(w) for w in words])) if words else 0.0,
        t0_avg_sent_len = n_w / max(len(sents), 1),
        t0_max_word_len = max((len(w) for w in words), default=0),
        t0_ttr          = len(cnt) / n_w,
        t0_hapax        = sum(1 for v in cnt.values() if v == 1) / n_w,
        t0_word_entropy = float(-(p * np.log2(p)).sum()),
        t0_char_entropy = float(-(pc * np.log2(pc)).sum()),
        t0_stop_ratio   = sum(1 for w in words if w in STOP) / n_w,
        t0_digit_ratio  = sum(c.isdigit() for c in t) / max(len(t), 1),
        t0_upper_ratio  = sum(c.isupper() for c in t) / max(len(t), 1),
        t0_punct_ratio  = sum(not c.isalnum() and not c.isspace() for c in t) / max(len(t), 1),
        t0_n_numbers    = len(nums),
        t0_num_density  = len(nums) / n_w,
        t0_max_number   = float(max([abs(float(x)) for x in nums], default=0.0)),
        t0_n_commas     = t.count(","),
        t0_comma_rate   = t.count(",") / max(len(sents), 1),
        t0_n_clause     = sum(words.count(c) for c in CLAUSE),
        t0_clause_rate  = sum(words.count(c) for c in CLAUSE) / max(len(sents), 1),
        t0_n_question   = t.count("?"),
        t0_n_newline    = t.count("\n"),
        # --- structural / arithmetic complexity
        t0_n_ops        = len(ops),
        t0_op_density   = len(ops) / n_w,
        t0_n_unique_num = len(set(nums)),
        t0_num_max_log  = float(np.log10(max(fnums) + 1)) if fnums else 0.0,
        t0_num_mean_log = float(np.mean([np.log10(abs(v) + 1) for v in fnums])) if fnums else 0.0,
        t0_num_spread   = float(np.std([np.log10(abs(v) + 1) for v in fnums])) if len(fnums) > 1 else 0.0,
        t0_n_decimal    = sum("." in x for x in nums),
        t0_n_large_num  = sum(abs(v) >= 1000 for v in fnums),
        t0_n_percent    = t.count("%"),
        # --- entity / discourse proxies
        t0_n_caps_tok   = len(caps),
        t0_caps_density = len(caps) / n_w,
        t0_long_word_r  = len(longw) / n_w,
        t0_syll_per_w   = syl / n_w,
        t0_flesch_kinc  = 0.39 * (n_w / max(len(sents), 1)) + 11.8 * (syl / n_w) - 15.59,
        t0_n_conj       = sum(words.count(c) for c in ["and", "or", "but", "then"]),
        t0_n_neg        = sum(words.count(c) for c in ["not", "no", "never", "none"]),
        t0_n_compare    = sum(words.count(c) for c in
                              ["more", "less", "than", "twice", "half", "each", "per"]),
        t0_n_temporal   = sum(words.count(c) for c in
                              ["after", "before", "then", "first", "next", "finally"]),
    )
    if _UNI:
        sur = [_UNI.get(w, _UNI_OOV) for w in words]
        f["t0_surprisal_mean"] = float(np.mean(sur))
        f["t0_surprisal_max"]  = float(np.max(sur))
        f["t0_surprisal_p90"]  = float(np.quantile(sur, .9))
    tk = _tok0()
    if tk is not None:
        n_tok = len(tk(t, add_special_tokens=False)["input_ids"])
    else:                                   # smoke proxy: subword split of whitespace words
        n_tok = int(round(n_w * 1.35 + f["t0_n_numbers"] * 1.2 + f["t0_punct_ratio"] * n_w))
    f["t0_n_tokens"]    = n_tok
    f["t0_frag_ratio"]  = n_tok / n_w                       # subwords per whitespace word
    f["t0_tok_per_char"]= n_tok / max(len(t), 1)
    if USE_SPACY:
        doc = _NLP(t)
        deps = [abs(tk.i - tk.head.i) for tk in doc if tk.head is not tk]
        def depth(tk, d=0):
            return d if tk.head is tk else depth(tk.head, d + 1)
        f["t0_dep_dist_mean"] = float(np.mean(deps)) if deps else 0.0
        f["t0_dep_dist_max"]  = float(np.max(deps)) if deps else 0.0
        f["t0_tree_depth"]    = float(np.mean([depth(tk) for tk in doc])) if len(doc) else 0.0
    return f

if USE_SPACY:
    import spacy; _NLP = spacy.load("en_core_web_sm")

# corpus-level unigram surprisal: how unusual is this prompt's vocabulary relative to the
# rest of the corpus? Built once, costs nothing at inference time.
_UNI, _UNI_OOV = {}, 0.0
_cnt = Counter(w for p in PROMPTS.prompt for w in re.findall(r"[A-Za-z\']+", p.lower()))
_tot = sum(_cnt.values())
_UNI = {w: -math.log2(c / _tot) for w, c in _cnt.items()}
_UNI_OOV = -math.log2(0.5 / _tot)

_src = ([item_text(t, p) for t, p in zip(PROMPTS.task, PROMPTS.prompt)]
        if TIER0_ON_ITEM else list(PROMPTS.prompt))
T0 = pd.DataFrame([tier0(p) for p in _src])
T0.insert(0, "uid", PROMPTS.uid.values)
save_parquet(T0, os.path.join(DIRS["feats"], "tier0.parquet"))
print("Tier-0:", T0.shape[1] - 1, "features")
T0.head(3)

Tier-0: 47 features


,uid,t0_n_chars,t0_n_words,t0_n_sents,t0_avg_word_len,t0_avg_sent_len,t0_max_word_len,t0_ttr,t0_hapax,t0_word_entropy,...,t0_n_conj,t0_n_neg,t0_n_compare,t0_n_temporal,t0_surprisal_mean,t0_surprisal_max,t0_surprisal_p90,t0_n_tokens,t0_frag_ratio,t0_tok_per_char
0,gsm8k-00000,259,43,5,4.372093,8.600000,8,0.790698,0.604651,4.990105,...,2,0,4,1,8.515974,16.619900,14.060608,65,1.511628,0.250965
1,gsm8k_direct-00000,264,43,6,4.488372,7.166667,8,0.767442,0.581395,4.926037,...,2,2,4,0,8.638015,16.619900,14.060608,64,1.488372,0.242424
2,gsm8k-00001,311,57,6,4.087719,9.500000,9,0.631579,0.438596,4.836504,...,1,0,0,1,7.996582,14.745431,12.807669,81,1.421053,0.260450


## 10. Tier-1 features — one forward pass, no generation

fp16 and the quantized model both fit on a T4 simultaneously at this scale (1.5B fp16 ≈ 3 GB,
nf4 ≈ 1.2 GB), so the KL divergence between their next-token distributions is computed
**exactly over the full vocabulary**, not from a top-k approximation.

Cost is one forward pass over the prompt — milliseconds, no tokens generated. The Tier-0 vs
Tier-1 comparison in §12 is the feature ablation that actually tells a practitioner something:
is free routing enough, or must you pay for a prefill?

In [57]:
def _dist_feats(lp_ref, lp_q):
    # lp_*: [T, V] log-softmax, float32 on GPU
    import torch
    p = lp_ref.exp()
    kl = (p * (lp_ref - lp_q)).sum(-1)                       # forward KL per position
    m = torch.logaddexp(lp_ref, lp_q) - math.log(2.0)
    js = 0.5 * ((p * (lp_ref - m)).sum(-1) +
                (lp_q.exp() * (lp_q - m)).sum(-1))
    ent_r = -(p * lp_ref).sum(-1); ent_q = -(lp_q.exp() * lp_q).sum(-1)
    a1 = lp_ref.argmax(-1); b1 = lp_q.argmax(-1)
    tr = lp_ref.topk(2, -1).values; tq = lp_q.topk(2, -1).values
    return dict(
        t1_kl_mean=kl.mean().item(), t1_kl_max=kl.max().item(),
        t1_kl_last=kl[-1].item(), t1_kl_p90=kl.quantile(0.9).item(),
        t1_kl_std=kl.std().item() if kl.numel() > 1 else 0.0,
        t1_js_mean=js.mean().item(), t1_js_last=js[-1].item(),
        t1_top1_agree=(a1 == b1).float().mean().item(),
        t1_top1_agree_last=float(a1[-1].item() == b1[-1].item()),
        t1_ent_ref_mean=ent_r.mean().item(), t1_ent_q_mean=ent_q.mean().item(),
        t1_dent_mean=(ent_q - ent_r).mean().item(),
        t1_ent_ref_last=ent_r[-1].item(), t1_ent_q_last=ent_q[-1].item(),
        t1_margin_ref_last=(tr[-1, 0] - tr[-1, 1]).item(),
        t1_margin_q_last=(tq[-1, 0] - tq[-1, 1]).item(),
        # --- quantized-only (deployable on the edge device without the fp16 model)
        t1q_ent_q_std=ent_q.std().item() if ent_q.numel() > 1 else 0.0,
        t1q_ent_q_max=ent_q.max().item(),
        t1q_margin_q_mean=(tq[:, 0] - tq[:, 1]).mean().item(),
        t1q_margin_q_min=(tq[:, 0] - tq[:, 1]).min().item(),
        t1q_top1_prob_q_mean=lp_q.max(-1).values.exp().mean().item(),
        t1q_low_conf_frac=(lp_q.max(-1).values.exp() < 0.5).float().mean().item(),
    )

def tier1(model_key, precision, chunk=128):
    key = f"tier1__{model_key}__{precision}__{GEN_TAG}"
    fn = os.path.join(DIRS["feats"], key + ".parquet")
    # A cached Tier-1 file is only valid if it covers EVERY current prompt and contains
    # every current feature column. Growing the corpus or adding features silently
    # invalidated this before, which produced a truncated design matrix and dropped the
    # edge-only feature group entirely.
    REQ_T1 = {"t1_kl_mean", "t1_nll_ref", "t1q_nll_q", "t1q_ent_q_std",
              "t1q_margin_q_mean", "t1q_low_conf_frac"}
    if os.path.exists(fn):
        d = read_parquet_safe(fn)
        if d is not None:
            miss = REQ_T1 - set(d.columns)
            if len(d) >= len(PROMPTS) and not miss:
                return d
            print(f"  [stale] {key}: rows {len(d)}/{len(PROMPTS)}, "
                  f"missing {sorted(miss) if miss else 'none'} -> regenerating")
            try: os.remove(fn)
            except OSError: pass
            try: shutil.rmtree(shard_dir(key))
            except OSError: pass
    if SMOKE:
        rng = np.random.RandomState(abs(hash(precision)) % (2**31))
        lat = PROMPTS._latent.values.astype(float)
        lat = (lat - lat.mean()) / (lat.std() + 1e-9)
        s = _PREC_DAMAGE.get(precision, 0.05) * 10
        d = pd.DataFrame(dict(
            uid=PROMPTS.uid.values,
            t1_kl_mean=np.abs(s * (0.4 + 0.5 * lat) + rng.normal(0, .3, len(lat))),
            t1_kl_max=np.abs(s * (1.0 + 0.8 * lat) + rng.normal(0, .6, len(lat))),
            t1_kl_last=np.abs(s * (0.5 + 0.6 * lat) + rng.normal(0, .4, len(lat))),
            t1_top1_agree=np.clip(1 - 0.05 * s * (0.5 + 0.4 * lat)
                                  + rng.normal(0, .03, len(lat)), 0, 1),
            t1_dent_mean=0.2 * s * lat + rng.normal(0, .2, len(lat)),
            t1_nll_ref=1.5 + 0.4 * lat + rng.normal(0, .2, len(lat)),
            t1_nll_q=1.5 + 0.4 * lat + 0.05 * s + rng.normal(0, .2, len(lat)),
            t1q_ent_q_std=np.abs(0.5 + 0.3 * lat + rng.normal(0, .2, len(lat))),
            t1q_margin_q_mean=2.0 - 0.5 * lat + rng.normal(0, .3, len(lat)),
            t1q_low_conf_frac=np.clip(.3 + .15 * lat + rng.normal(0, .05, len(lat)), 0, 1),
        ))
        d["t1q_nll_q"] = d.t1_nll_q
        d["t1_dnll"] = d.t1_nll_q - d.t1_nll_ref
        save_parquet(d, fn); return d

    import torch
    # resume: each shard stores a JSON blob of feature dicts keyed by uid
    cached = {}
    for f in sorted(os.listdir(shard_dir(key))):
        p = read_parquet_safe(os.path.join(shard_dir(key), f))
        if p is not None:
            cached.update({u: json.loads(o) for u, o in zip(p.uid, p.output)})
    todo = PROMPTS[~PROMPTS.uid.isin(cached)]
    if len(cached): print(f"  [resume] {key}: {len(cached)} cached, {len(todo)} remaining")
    if not len(todo):
        d = pd.DataFrame(list(cached.values())); save_parquet(d, fn); return d

    mr, tok = load_model(model_key, REF_PREC)
    mq, _   = load_model(model_key, precision)
    rows, buf = [], {}
    shard_i = len([f for f in os.listdir(shard_dir(key)) if f.endswith(".parquet")])
    for i, (uid, ptxt) in enumerate(zip(todo.uid, todo.prompt)):
        enc = tok(chatify(tok, ptxt), return_tensors="pt", truncation=True,
                  max_length=MAX_LEN).to("cuda")
        with torch.no_grad():
            lr = mr(**enc).logits[0].float()
            lq = mq(**enc).logits[0].float()
        ids = enc["input_ids"][0]
        nll_r = torch.nn.functional.cross_entropy(lr[:-1], ids[1:]).item()
        nll_q = torch.nn.functional.cross_entropy(lq[:-1], ids[1:]).item()
        f = {}
        for st in range(0, lr.shape[0], chunk):                # chunked to bound memory
            chunk_feats = _dist_feats(torch.log_softmax(lr[st:st+chunk], -1),
                                      torch.log_softmax(lq[st:st+chunk], -1))
            for k, v in chunk_feats.items(): f.setdefault(k, []).append(v)
        f = {k: float(np.mean(v)) for k, v in f.items()}
        f.update(uid=uid, t1_nll_ref=nll_r, t1_nll_q=nll_q, t1_dnll=nll_q - nll_r,
                 t1q_nll_q=nll_q)
        rows.append(f); buf[uid] = json.dumps(f)
        if len(buf) >= SAVE_EVERY or i + 1 == len(todo):
            # reuse the same shard writer as generation: one code path, not two
            write_shard(key, buf, shard_i)
            buf = {}; shard_i += 1
            print(f"  tier1 {precision} {i+1}/{len(todo)}", flush=True)
        del lr, lq, enc, ids
    del mr, mq, tok
    free()
    cached.update({r["uid"]: r for r in rows})
    d = pd.DataFrame(list(cached.values())); save_parquet(d, fn); return d

if SKIP_TIER1:
    T1 = {p: pd.DataFrame({"uid": PROMPTS.uid}) for p in PRECISIONS if p != REF_PREC}
    print("SKIP_TIER1=True -- Tier-1 features skipped; tier0-only arms will run in S12")
else:
    T1 = {p: tier1(PRIMARY, p) for p in PRECISIONS if p != REF_PREC}
print({k: v.shape for k, v in T1.items()})

{'nf4': (5372, 27), 'fp4': (5372, 27), 'int8': (5372, 27)}


## 11. Design matrix

One matrix per quantization tier. `fp16_selfrep` is carried through the identical pipeline as a
**negative control**: features should *not* predict GPU nondeterminism. If they do, something in
the pipeline is leaking and the main result is void.

In [58]:
def design(precision, model_key=PRIMARY):
    lab = LAB[(LAB.model == model_key) & (LAB.precision == precision)]
    src = precision if precision in T1 else list(T1)[0]
    X = lab.merge(T0, on="uid").merge(T1[src], on="uid", suffixes=("", "_t1"))
    return X

DESIGNS = {p: design(p) for p in list(T1) + ["fp16_selfrep"]}
F_T0   = [c for c in T0.columns if c.startswith("t0_")]
_dcols = DESIGNS[list(T1)[0]].columns
F_T1   = [c for c in _dcols if c.startswith("t1_") or c.startswith("t1q_")]
# Edge tier: computable on-device from the QUANTIZED model alone -- no fp16 reference
# needed at deployment. This is the practically deployable middle ground between the
# free Tier-0 stats and the paired Tier-1 divergences.
F_EDGE = [c for c in _dcols if c.startswith("t1q_")]
print({k: (v.shape, round(v.y.mean(), 3)) for k, v in DESIGNS.items()})
print(f"Tier-0: {len(F_T0)} | Tier-1 (paired): {len(F_T1)} | Edge-only: {len(F_EDGE)}")
if not F_EDGE:
    print("\n*** WARNING: no edge-only features found. The deployable arm cannot run. ***")
    print("    Delete the Tier-1 caches and re-run S10 -- see the [stale] check there.")
else:
    print(f"    Edge-only features: {F_EDGE}")
    print("    'edge' and 'tier0+edge' arms answer the deployability question:")
    print("    can fragility be predicted WITHOUT running the fp16 reference?")

{'nf4': ((3625, 78), np.float64(0.119)), 'fp4': ((3625, 78), np.float64(0.172)), 'int8': ((3625, 78), np.float64(0.036)), 'fp16_selfrep': ((3625, 78), np.float64(0.002))}
Tier-0: 47 | Tier-1 (paired): 26 | Edge-only: 7
    Edge-only features: ['t1q_ent_q_std', 't1q_ent_q_max', 't1q_margin_q_mean', 't1q_margin_q_min', 't1q_top1_prob_q_mean', 't1q_low_conf_frac', 't1q_nll_q']
    'edge' and 'tier0+edge' arms answer the deployability question:
    can fragility be predicted WITHOUT running the fp16 reference?


## 12. Meta-classifiers and baselines

In [59]:
from sklearn.model_selection import (RepeatedStratifiedKFold, cross_val_predict,
                                     StratifiedKFold, GroupKFold)
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             roc_curve, precision_recall_curve)
from scipy import stats
from xgboost import XGBClassifier
from sklearn.ensemble import (RandomForestClassifier, HistGradientBoostingClassifier,
                              StackingClassifier)
from sklearn.model_selection import RandomizedSearchCV

def stacker(y=None):
    # heterogeneous ensemble: trees capture interactions, the linear model captures
    # smooth monotone effects. Logistic meta-learner on out-of-fold predictions.
    return StackingClassifier(
        estimators=[("xgb", xgb(y)),
                    ("rf", RandomForestClassifier(n_estimators=400, min_samples_leaf=3,
                                                  class_weight="balanced_subsample",
                                                  n_jobs=4, random_state=SEED)),
                    ("lr", logreg())],
        final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced"),
        cv=3, n_jobs=1, passthrough=False)

HP_GRID = dict(n_estimators=[200, 300, 500, 800],
               max_depth=[2, 3, 4, 5, 6],
               learning_rate=[0.02, 0.04, 0.07, 0.12],
               subsample=[0.7, 0.85, 1.0],
               colsample_bytree=[0.5, 0.7, 0.85, 1.0],
               min_child_weight=[1, 3, 6, 10],
               reg_lambda=[0.5, 1.5, 5.0, 15.0])

def xgb_searched(y):
    # Randomized search wrapped so it can be fitted INSIDE each outer CV fold --
    # tuning on the training split only, never on the fold being scored.
    return RandomizedSearchCV(xgb(y), HP_GRID, n_iter=HP_ITERS, scoring="roc_auc",
                              cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
                              random_state=SEED, n_jobs=4, refit=True)

def xgb(y=None):
    spw = 1.0
    if y is not None and y.sum() > 0: spw = float((len(y) - y.sum()) / y.sum())
    return XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.07,
                         subsample=0.85, colsample_bytree=0.85, reg_lambda=1.5,
                         eval_metric="logloss", n_jobs=4, random_state=SEED,
                         scale_pos_weight=spw)

def logreg():
    return make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=2000, class_weight="balanced"))

def build_arms(D):
    y = D.y.values
    task1h = pd.get_dummies(D.task, prefix="task").astype(float)
    arms = {
        "majority"     : (DummyClassifier(strategy="prior"), task1h.values),
        "task_id_only" : (logreg(),  task1h.values),                       # CONTROL
        "tier0"        : (xgb(y),    D[F_T0].values),
        "tier0+tier1"  : (xgb(y),    D[F_T0 + F_T1].values),
        "tier0+task"   : (xgb(y),    np.hstack([D[F_T0].values, task1h.values])),
        "length_rule"  : (logreg(),  D[["t0_n_words"]].values),          # CONTROL
        "linear_all"   : (logreg(),  D[F_T0 + F_T1].values),
        "rf_tier0"     : (RandomForestClassifier(n_estimators=500, min_samples_leaf=3,
                                                 class_weight="balanced_subsample",
                                                 n_jobs=4, random_state=SEED),
                          D[F_T0].values),
        "hgb_tier0"    : (HistGradientBoostingClassifier(max_iter=300, learning_rate=.07,
                                                         max_leaf_nodes=15,
                                                         l2_regularization=1.0,
                                                         random_state=SEED),
                          D[F_T0].values),
    }
    if F_T1:
        arms["tier1"] = (xgb(y), D[F_T1].values)
    if F_EDGE:
        # deployable: quantized model only, no fp16 reference needed at run time
        arms["tier0+edge"] = (xgb(y), D[F_T0 + F_EDGE].values)
        arms["edge"]       = (xgb(y), D[F_EDGE].values)
    if not SMOKE:
        arms["stack"] = (stacker(y), D[F_T0 + F_T1].values)
    if USE_EMB:
        arms["embedding"] = (logreg(), EMB.set_index("uid").loc[D.uid].values)
    if HP_SEARCH:
        arms["tier0_tuned"] = (xgb_searched(y), D[F_T0].values)          # HEADLINE
        if F_T1:
            arms["tier0+tier1_tuned"] = (xgb_searched(y), D[F_T0 + F_T1].values)
    return arms

def cv_eval(D, seed=SEED):
    y = D.y.values
    cv    = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=seed)
    cheap = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=1, random_state=seed)
    full_splits  = [(tr, te) for tr, te in cv.split(np.zeros(len(y)), y)]
    cheap_splits = [(tr, te) for tr, te in cheap.split(np.zeros(len(y)), y)]
    # cheap must be a prefix of full, otherwise the paired test below is not paired
    assert all(np.array_equal(a[1], b[1])
               for a, b in zip(full_splits, cheap_splits)), "split prefix mismatch"

    rows, oof, folds, nfold = [], {}, {}, {}
    for name, (clf, X) in build_arms(D).items():
        expensive = ("tuned" in name) or (name == "stack")
        splits = cheap_splits if expensive else full_splits
        auc, ap, br = [], [], []
        for tr, te in splits:
            import copy; c = copy.deepcopy(clf); c.fit(X[tr], y[tr])
            if hasattr(c, "best_estimator_"): c = c.best_estimator_
            s = c.predict_proba(X[te])[:, 1]
            auc.append(roc_auc_score(y[te], s)); ap.append(average_precision_score(y[te], s))
            br.append(brier_score_loss(y[te], s))
        folds[name] = np.array(auc); nfold[name] = len(auc)
        n = len(auc); ci = stats.t.ppf(.975, n - 1) * np.std(auc, ddof=1) / np.sqrt(n)
        rows.append(dict(arm=name, auc=np.mean(auc), auc_ci=ci, auc_sd=np.std(auc, ddof=1),
                         ap=np.mean(ap), brier=np.mean(br), n_folds=n,
                         pos_rate=y.mean(), n=len(y)))
        try:
            oof[name] = cross_val_predict(clf, X, y, method="predict_proba",
                                          cv=StratifiedKFold(N_SPLITS, shuffle=True,
                                                             random_state=seed))[:, 1]
        except Exception as e:
            print(f"  [warn] oof failed for {name}: {type(e).__name__}"); oof[name] = None
    oof = {k: v for k, v in oof.items() if v is not None}
    R = pd.DataFrame(rows).sort_values("auc", ascending=False).reset_index(drop=True)

    # Paired test vs the task-ID control, on the folds the two arms actually share.
    # cheap_splits is a prefix of full_splits (asserted above), so truncating the
    # control to the shorter arm's length pairs fold-for-fold.
    best = R.arm.iloc[0]
    k = min(nfold[best], nfold["task_id_only"])
    a, b = folds[best][:k], folds["task_id_only"][:k]
    try:
        t, p = stats.ttest_rel(a, b)
        w = stats.wilcoxon(a, b).pvalue if k >= 6 else np.nan
    except Exception:
        t, p, w = np.nan, np.nan, np.nan
    R.attrs.update(best=best, t=float(t), p=float(p), p_wilcoxon=float(w),
                   n_paired=k, delta=float(np.mean(a - b)))
    return R, oof, folds

CV = {}
for prec, D in DESIGNS.items():
    if D.y.sum() < 20:
        if prec == "fp16_selfrep":
            # Too few positives to MODEL is the ideal outcome here, not a failure:
            # it means run-to-run nondeterminism is almost absent. Report the rate.
            print(f"[NEGATIVE CONTROL] fp16-vs-fp16 disagreement: {int(D.y.sum())}/{len(D)} "
                  f"= {D.y.mean()*100:.2f}% -- too few positives to fit a classifier, "
                  f"which is the strongest possible form of this control.")
        else:
            print(f"[skip] {prec}: only {int(D.y.sum())} positives")
        continue
    R, oof, folds = cv_eval(D)
    CV[prec] = dict(R=R, oof=oof, folds=folds)
    print(f"\n=== {prec} (positives={int(D.y.sum())}/{len(D)}) ===")
    print(R.round(4).to_string(index=False))
    print(f"best={R.attrs['best']} vs task_id_only: t={R.attrs['t']:.2f}, p={R.attrs['p']:.2e}")
    R.to_csv(os.path.join(DIRS["results"], f"cv_{prec}.csv"), index=False)
    np.save(os.path.join(DIRS["results"], f"oof_{prec}.npy"),
            np.vstack([oof[k] for k in sorted(oof)]))
    json.dump(sorted(oof), open(os.path.join(DIRS["results"], f"oof_{prec}_arms.json"), "w"))


=== nf4 (positives=430/3625) ===
              arm    auc  auc_ci  auc_sd     ap  brier  n_folds  pos_rate    n
tier0+tier1_tuned 0.8674  0.0224  0.0180 0.4866 0.1256        5    0.1186 3625
      tier0+tier1 0.8605  0.0055  0.0133 0.4684 0.0965       25    0.1186 3625
            tier1 0.8579  0.0056  0.0135 0.4695 0.1022       25    0.1186 3625
       linear_all 0.8558  0.0058  0.0141 0.4587 0.1501       25    0.1186 3625
            stack 0.8435  0.0244  0.0197 0.4493 0.1557        5    0.1186 3625
      tier0_tuned 0.7383  0.0371  0.0299 0.2926 0.1937        5    0.1186 3625
     task_id_only 0.7221  0.0115  0.0280 0.2340 0.2132       25    0.1186 3625
         rf_tier0 0.7216  0.0103  0.0250 0.2871 0.1021       25    0.1186 3625
       tier0+task 0.7052  0.0083  0.0200 0.2704 0.1268       25    0.1186 3625
       tier0+edge 0.6995  0.0075  0.0181 0.2574 0.1251       25    0.1186 3625
        hgb_tier0 0.6994  0.0085  0.0205 0.2582 0.1025       25    0.1186 3625
      length_rule 

### 12c. Persist the fitted predictors

Retraining is seconds, but the fitted router is a paper artifact and reviewers increasingly ask
for it. Saved as JSON (portable across xgboost versions) rather than pickle.

In [60]:
for prec in [p for p in CV if p != "fp16_selfrep"]:
    D = DESIGNS[prec]
    for arm, feats in [("tier0", F_T0), ("tier0+tier1", F_T0 + F_T1)]:
        m = xgb(D.y.values); m.fit(D[feats].values, D.y.values)
        m.save_model(os.path.join(DIRS["models"], f"router_{prec}_{arm}.json"))
        json.dump(feats, open(os.path.join(DIRS["models"],
                              f"router_{prec}_{arm}_features.json"), "w"))
print("saved routers ->", DIRS["models"])

saved routers -> /content/drive/MyDrive/qdegrade/models


### 12e. Paired arm comparison

Two arms differing by 0.02 AUC is noise unless the comparison is paired on identical folds.
Every arm here is scored on the same `RepeatedStratifiedKFold` splits, so differences can be
tested with a Wilcoxon signed-rank test on the per-fold AUCs.

What to read:
- **vs `task_id_only`** and **vs `length_rule`**: the two comparisons that decide acceptance.
- **`linear_all` vs the tree arms**: if the linear model matches, the mapping from prompt
  features to fragility is essentially simple, and the deployable router can be a logistic
  regression. That is a stronger deployment story, not a weaker result.
- **tree vs tree** (`tier0` / `rf_tier0` / `hgb_tier0`): expect near-ties. Report it in one
  small table and do not build the paper on it.

In [61]:
from scipy.stats import wilcoxon

def compare_arms(folds, ref="tier0"):
    n = {k: len(v) for k, v in folds.items()}
    common = max(set(n.values()), key=list(n.values()).count)   # arms on identical splits
    use = {k: v for k, v in folds.items() if n[k] == common}
    if ref not in use: ref = max(use, key=lambda k: use[k].mean())
    rows = []
    for k, v in use.items():
        if k == ref: continue
        d = use[ref] - v
        try:   p = wilcoxon(use[ref], v).pvalue
        except Exception: p = np.nan
        lo, hi = np.percentile([np.mean(np.random.choice(d, len(d))) for _ in range(2000)],
                               [2.5, 97.5])
        rows.append(dict(reference=ref, arm=k, auc_arm=v.mean(), auc_ref=use[ref].mean(),
                         delta=d.mean(), ci_lo=lo, ci_hi=hi, p_wilcoxon=p,
                         n_folds=common,
                         verdict=("reference better" if p < .05 and d.mean() > 0 else
                                  "arm better" if p < .05 else "no difference")))
    return pd.DataFrame(rows).sort_values("delta", ascending=False)

CMP = {}
for prec in [p for p in CV if p != "fp16_selfrep"]:
    CMP[prec] = compare_arms(CV[prec]["folds"])
    print(f"\n=== {prec}: paired comparison against the reference arm ===")
    print(CMP[prec].round(4).to_string(index=False))


=== nf4: paired comparison against the reference arm ===
reference          arm  auc_arm  auc_ref   delta   ci_lo   ci_hi  p_wilcoxon  n_folds          verdict
    tier0     majority   0.5000   0.6954  0.1954  0.1877  0.2031      0.0000       25 reference better
    tier0         edge   0.6857   0.6954  0.0097 -0.0019  0.0213      0.0851       25    no difference
    tier0  length_rule   0.6991   0.6954 -0.0037 -0.0143  0.0063      0.5782       25    no difference
    tier0    hgb_tier0   0.6994   0.6954 -0.0040 -0.0090  0.0008      0.2002       25    no difference
    tier0   tier0+edge   0.6995   0.6954 -0.0041 -0.0101  0.0018      0.2411       25    no difference
    tier0   tier0+task   0.7052   0.6954 -0.0098 -0.0132 -0.0062      0.0000       25       arm better
    tier0     rf_tier0   0.7216   0.6954 -0.0261 -0.0328 -0.0194      0.0000       25       arm better
    tier0 task_id_only   0.7221   0.6954 -0.0266 -0.0357 -0.0177      0.0000       25       arm better
    tier0   lin

### 12d. Two label variants that raise the ceiling

**Pooled model.** Train one predictor across all quantization tiers with bit-width as a
feature. This roughly doubles the positives, which stabilises the estimate, and it is also
the more realistic deployment: one router serving whatever precision the device runs.

**Consistent fragility.** The per-tier binary label is noisy at the margin -- a prompt that
barely survives fp16 can flip for reasons unrelated to its structure. Restricting to prompts
that degrade under **every** quantized tier (positives) versus survive under **all** of them
(negatives), and dropping the ambiguous middle, gives a cleaner target. Report the excluded
fraction honestly; this is a secondary analysis, not a replacement for the primary result.

In [62]:
# ---- pooled across tiers, bit-width as a feature
_qp = [p for p in DESIGNS if p != "fp16_selfrep"]
BITS = {"int8": 8.0, "nf4": 4.0, "fp4": 4.0}
POOL = pd.concat([DESIGNS[p].assign(prec=p, bits=BITS.get(p, 4.0),
                                    is_fp4=float(p == "fp4")) for p in _qp],
                 ignore_index=True)
Fp = F_T0 + ["bits", "is_fp4"]
yv = POOL.y.values
cvp = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=3, random_state=SEED)
auc = []
for tr, te in cvp.split(POOL[Fp].values, yv):
    c = xgb(yv[tr]); c.fit(POOL[Fp].values[tr], yv[tr])
    auc.append(roc_auc_score(yv[te], c.predict_proba(POOL[Fp].values[te])[:, 1]))
ci = stats.t.ppf(.975, len(auc) - 1) * np.std(auc, ddof=1) / np.sqrt(len(auc))
POOLED = dict(n=len(POOL), pos=int(yv.sum()), auc=float(np.mean(auc)), ci=float(ci))
print(f"[pooled] n={POOLED['n']} pos={POOLED['pos']} "
      f"AUC={POOLED['auc']:.3f} +/- {POOLED['ci']:.3f}")

# ---- consistent fragility across every quantized tier
wide = None
for p in _qp:
    d = DESIGNS[p][["uid", "task", "y"]].rename(columns={"y": p})
    wide = d if wide is None else wide.merge(d, on=["uid", "task"], how="inner")
CONS = None
if wide is not None and len(_qp) > 1:
    ycols = [p for p in _qp]
    allpos = wide[ycols].all(axis=1); allneg = (~wide[ycols].astype(bool)).all(axis=1)
    keep = allpos | allneg
    W = wide[keep].copy(); W["y"] = allpos[keep].astype(int)
    W = W.merge(T0, on="uid")
    print(f"[consistent] kept {len(W)}/{len(wide)} ({100*len(W)/len(wide):.0f}%), "
          f"{int(W.y.sum())} always-fragile")
    if 20 < W.y.sum() < len(W) - 20:
        a = []
        for tr, te in RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=3,
                                              random_state=SEED).split(W[F_T0].values, W.y.values):
            c = xgb(W.y.values[tr]); c.fit(W[F_T0].values[tr], W.y.values[tr])
            a.append(roc_auc_score(W.y.values[te], c.predict_proba(W[F_T0].values[te])[:, 1]))
        ci = stats.t.ppf(.975, len(a) - 1) * np.std(a, ddof=1) / np.sqrt(len(a))
        CONS = dict(n=len(W), pos=int(W.y.sum()), kept_frac=len(W) / len(wide),
                    auc=float(np.mean(a)), ci=float(ci))
        print(f"[consistent] AUC={CONS['auc']:.3f} +/- {CONS['ci']:.3f}")
        # within-task, since this is where the confound bites
        oofc = cross_val_predict(xgb(W.y.values), W[F_T0].values, W.y.values,
                                 method="predict_proba",
                                 cv=StratifiedKFold(N_SPLITS, shuffle=True,
                                                    random_state=SEED))[:, 1]
        for t in sorted(W.task.unique()):
            m = (W.task == t).values
            if 0 < W.y.values[m].sum() < m.sum():
                print(f"    within {t}: AUC={roc_auc_score(W.y.values[m], oofc[m]):.3f} "
                      f"(n={m.sum()}, pos={int(W.y.values[m].sum())})")

[pooled] n=10875 pos=1187 AUC=0.825 +/- 0.007
[consistent] kept 2837/3625 (78%), 55 always-fragile
[consistent] AUC=0.799 +/- 0.030
    within arc_challenge: AUC=0.631 (n=575, pos=6)
    within arc_easy: AUC=0.474 (n=1157, pos=6)
    within gsm8k: AUC=0.652 (n=341, pos=34)
    within gsm8k_direct: AUC=0.395 (n=24, pos=7)
    within sst2: AUC=0.699 (n=740, pos=2)


## 13. The three confound controls

This section decides whether the paper is accepted. If `tier0` does not beat `task_id_only`,
and if within-task AUC collapses to chance, then the classifier learned *which dataset a prompt
came from*, not *how fragile it is*. Report all of this in the main table, not an appendix.

In [63]:
def within_task_auc(D, oof):
    rows = []
    for arm, s in oof.items():
        r = {"arm": arm}
        for task in sorted(D.task.unique()):
            m = (D.task == task).values
            r[task] = (roc_auc_score(D.y.values[m], s[m])
                       if 0 < D.y.values[m].sum() < m.sum() else np.nan)
        r["macro"] = np.nanmean([r[t] for t in sorted(D.task.unique())])
        rows.append(r)
    return pd.DataFrame(rows).sort_values("macro", ascending=False)

def loto(D):
    y, g = D.y.values, D.task.values
    rows = []
    for name, feats in ([("tier0", F_T0)] +
                        ([("tier1", F_T1), ("tier0+tier1", F_T0 + F_T1)] if F_T1 else [])):
        X = D[feats].values; per = {}
        for tr, te in GroupKFold(n_splits=D.task.nunique()).split(X, y, groups=g):
            held = g[te][0]
            if not (0 < y[te].sum() < len(te)): per[held] = np.nan; continue
            c = xgb(y[tr]); c.fit(X[tr], y[tr])
            per[held] = roc_auc_score(y[te], c.predict_proba(X[te])[:, 1])
        per["arm"] = name; per["macro"] = np.nanmean([v for k, v in per.items() if k != "arm"])
        rows.append(per)
    return pd.DataFrame(rows)

WT, LT = {}, {}
for prec in CV:
    D = DESIGNS[prec]
    WT[prec] = within_task_auc(D, CV[prec]["oof"])
    LT[prec] = loto(D)
    print(f"\n--- {prec} :: within-task AUC ---"); print(WT[prec].round(3).to_string(index=False))
    print(f"--- {prec} :: leave-one-task-out AUC ---"); print(LT[prec].round(3).to_string(index=False))

D_nc = DESIGNS.get("fp16_selfrep")
if "fp16_selfrep" in CV:
    a = CV["fp16_selfrep"]["R"].auc.max()
    print(f"\n[NEGATIVE CONTROL] best AUC on fp16-vs-fp16 nondeterminism = {a:.3f}")
    print("  Expect ~0.5-0.6. Much higher means the pipeline is leaking.")
elif D_nc is not None:
    print(f"\n[NEGATIVE CONTROL] fp16-vs-fp16 disagreement = "
          f"{int(D_nc.y.sum())}/{len(D_nc)} ({D_nc.y.mean()*100:.2f}%). "
          f"Signal-to-noise vs each quantized tier:")
    for p in [q for q in DESIGNS if q != "fp16_selfrep"]:
        print(f"    {p}: {DESIGNS[p].y.mean()/max(D_nc.y.mean(),1e-9):.0f}x the noise floor")


--- nf4 :: within-task AUC ---
              arm  arc_challenge  arc_easy  gsm8k  gsm8k_direct  sst2  macro
tier0+tier1_tuned          0.832     0.869  0.612         0.628 0.987  0.785
      tier0+tier1          0.830     0.858  0.570         0.629 0.985  0.774
            tier1          0.825     0.863  0.557         0.612 0.993  0.770
       linear_all          0.824     0.869  0.575         0.700 0.882  0.770
            stack          0.818     0.852  0.555         0.632 0.793  0.730
         rf_tier0          0.587     0.573  0.654         0.590 0.501  0.581
      length_rule          0.594     0.607  0.645         0.584 0.362  0.558
      tier0_tuned          0.614     0.583  0.654         0.511 0.395  0.551
       tier0+task          0.584     0.546  0.616         0.536 0.430  0.543
            tier0          0.566     0.554  0.633         0.508 0.418  0.536
       tier0+edge          0.551     0.555  0.611         0.489 0.423  0.526
        hgb_tier0          0.592     0.559  

## 14. Feature-group ablation

In [64]:
GROUPS = {
    "length"  : [c for c in F_T0 if any(k in c for k in ["n_chars","n_words","n_sents",
                                                          "sent_len","word_len","newline",
                                                          "n_tokens","frag_ratio",
                                                          "tok_per_char"])],
    "lexical" : [c for c in F_T0 if any(k in c for k in ["ttr","hapax","stop","entropy"])],
    "numeric" : [c for c in F_T0 if any(k in c for k in ["digit","number"])],
    "structure":[c for c in F_T0 if any(k in c for k in ["comma","clause","question","punct",
                                                          "upper","dep_","tree_"])],
    "arith"   : [c for c in F_T0 if any(k in c for k in ["ops","num_","n_decimal",
                                                         "large_num","percent","unique_num"])],
    "discourse":[c for c in F_T0 if any(k in c for k in ["conj","neg","compare","temporal",
                                                         "caps","flesch","syll","surprisal"])],
    "t1_div"  : [c for c in F_T1 if any(k in c for k in ["kl","js","agree"])],
    "t1_unc"  : [c for c in F_T1 if any(k in c for k in ["ent","margin","nll","conf"])],
}

def ablate(D, seed=SEED):
    y = D.y.values; full = F_T0 + F_T1
    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=2, random_state=seed)
    def score(feats):
        X = D[feats].values; a = []
        for tr, te in cv.split(X, y):
            c = xgb(y[tr]); c.fit(X[tr], y[tr])
            a.append(roc_auc_score(y[te], c.predict_proba(X[te])[:, 1]))
        return float(np.mean(a)), float(np.std(a, ddof=1))
    base, base_sd = score(full)
    rows = [dict(removed="none (full)", auc=base, sd=base_sd, delta=0.0, n_feat=len(full))]
    for g, cols in GROUPS.items():
        keep = [c for c in full if c not in cols]
        if not keep: continue
        a, sd = score(keep)
        rows.append(dict(removed=g, auc=a, sd=sd, delta=a - base, n_feat=len(keep)))
    return pd.DataFrame(rows)

ABL = {p: ablate(DESIGNS[p]) for p in CV if p != "fp16_selfrep"}
for p, a in ABL.items():
    print(f"\n--- {p} ---"); print(a.round(4).to_string(index=False))

IMP = {}
for p in ABL:
    D = DESIGNS[p]; c = xgb(D.y.values); c.fit(D[F_T0 + F_T1].values, D.y.values)
    IMP[p] = (pd.Series(c.feature_importances_, index=F_T0 + F_T1)
                .sort_values(ascending=False))


--- nf4 ---
    removed    auc     sd   delta  n_feat
none (full) 0.8572 0.0140  0.0000      73
     length 0.8604 0.0119  0.0032      63
    lexical 0.8562 0.0135 -0.0010      68
    numeric 0.8579 0.0134  0.0007      70
  structure 0.8588 0.0139  0.0016      66
      arith 0.8592 0.0140  0.0020      64
  discourse 0.8594 0.0116  0.0022      62
     t1_div 0.8238 0.0146 -0.0334      64
     t1_unc 0.8508 0.0157 -0.0063      57

--- fp4 ---
    removed    auc     sd   delta  n_feat
none (full) 0.9216 0.0098  0.0000      73
     length 0.9197 0.0104 -0.0019      63
    lexical 0.9210 0.0099 -0.0006      68
    numeric 0.9213 0.0100 -0.0003      70
  structure 0.9235 0.0090  0.0019      66
      arith 0.9217 0.0101  0.0001      64
  discourse 0.9228 0.0107  0.0012      62
     t1_div 0.8560 0.0090 -0.0656      64
     t1_unc 0.9150 0.0060 -0.0066      57

--- int8 ---
    removed    auc     sd   delta  n_feat
none (full) 0.8057 0.0270  0.0000      73
     length 0.8048 0.0265 -0.0009   

## 15. Router simulation — the headline result

Every row here is fp16-correct, so *parity retention* = the fraction of the fp16 reference
accuracy the deployed system keeps. Routing a prompt to fp16 retains it by definition; leaving
it on the quantized model retains it iff `y=0`.

Report an operating point in the abstract, e.g. *"escalating 15% of prompts recovers 78% of the
accuracy lost to 4-bit quantization."* That single sentence is what makes this a systems paper
rather than a classification exercise.

In [65]:
def router_curve(D, scores, grid=np.linspace(0, 1, 51)):
    y = D.y.values; n = len(y); order = np.argsort(-scores)
    base = 1 - y.mean()
    out = []
    for c in grid:
        k = int(round(c * n))
        routed = np.zeros(n, bool); routed[order[:k]] = True
        acc = (routed | (y == 0)).mean()
        oracle_k = min(k, int(y.sum()))
        out.append(dict(coverage=c,
                        model=acc,
                        random=base + c * (1 - base),
                        oracle=(n - y.sum() + oracle_k) / n,
                        always_quant=base, always_fp16=1.0,
                        recovered=(acc - base) / max(1 - base, 1e-9)))
    return pd.DataFrame(out)

ROUTE = {}
for p in CV:
    if p == "fp16_selfrep": continue
    D = DESIGNS[p]
    for arm in ["tier0", "tier0+tier1"]:
        ROUTE[(p, arm)] = router_curve(D, CV[p]["oof"][arm])

OPS = []
for (p, arm), c in ROUTE.items():
    for cov in [0.05, 0.10, 0.15, 0.25]:
        r = c.iloc[(c.coverage - cov).abs().idxmin()]
        OPS.append(dict(precision=p, arm=arm, coverage=cov, parity=r.model,
                        random=r["random"], oracle=r.oracle,
                        lost_recovered=r.recovered, baseline=r.always_quant))
OPS = pd.DataFrame(OPS)
print(OPS.round(3).to_string(index=False))

precision         arm  coverage  parity  random  oracle  lost_recovered  baseline
      nf4       tier0      0.05   0.902   0.888   0.942           0.177     0.881
      nf4       tier0      0.10   0.912   0.893   0.981           0.256     0.881
      nf4       tier0      0.15   0.919   0.898   1.000           0.321     0.881
      nf4       tier0      0.25   0.939   0.910   1.000           0.484     0.881
      nf4 tier0+tier1      0.05   0.917   0.888   0.942           0.302     0.881
      nf4 tier0+tier1      0.10   0.926   0.893   0.981           0.377     0.881
      nf4 tier0+tier1      0.15   0.936   0.898   1.000           0.463     0.881
      nf4 tier0+tier1      0.25   0.961   0.910   1.000           0.670     0.881
      fp4       tier0      0.05   0.859   0.838   0.888           0.181     0.828
      fp4       tier0      0.10   0.876   0.845   0.927           0.283     0.828
      fp4       tier0      0.15   0.896   0.852   0.968           0.395     0.828
      fp4       

## 16. Deployment cost: router overhead, latency, memory

The routing claim is only honest if the router is cheap relative to what it saves. Measure,
do not assert. On CPU-only Colab the model timings are skipped and only the router overhead is
reported, which is still the number that matters most.

In [66]:
import timeit
COST = {}
_p = list(PROMPTS.prompt[:200])
COST["tier0_extract_ms"] = 1000 * timeit.timeit(lambda: [tier0(x) for x in _p], number=1) / len(_p)

if CV:
    prec0 = [p for p in CV if p != "fp16_selfrep"][0]
    D0 = DESIGNS[prec0]; c0 = xgb(D0.y.values); c0.fit(D0[F_T0].values, D0.y.values)
    Xb = D0[F_T0].values[:200]
    COST["xgb_predict_ms"] = 1000 * timeit.timeit(lambda: c0.predict_proba(Xb), number=5) / (5*200)
COST["router_total_ms"] = COST["tier0_extract_ms"] + COST.get("xgb_predict_ms", 0.0)

if not SMOKE:
    import torch
    for prec in PRECISIONS:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        m, tok = load_model(PRIMARY, prec)
        enc = tok(chatify(tok, _p[0]), return_tensors="pt").to("cuda")
        with torch.no_grad(): m(**enc)                                  # warmup
        torch.cuda.synchronize(); t0 = time.time()
        with torch.no_grad():
            for _ in range(10): m(**enc)
        torch.cuda.synchronize()
        COST[f"prefill_ms_{prec}"] = 1000 * (time.time() - t0) / 10
        COST[f"peak_vram_gb_{prec}"] = torch.cuda.max_memory_allocated() / 1e9
        del enc, m, tok          # caller releases first; free() only sweeps up after
        free()

COST["overhead_vs_prefill_pct"] = (100 * COST["router_total_ms"] /
                                   COST.get(f"prefill_ms_{REF_PREC}", np.nan))
print(json.dumps({k: round(v, 4) for k, v in COST.items()}, indent=2))
json.dump(COST, open(os.path.join(DIRS["results"], "cost.json"), "w"), indent=2)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{
  "tier0_extract_ms": 1.1125,
  "xgb_predict_ms": 0.0045,
  "router_total_ms": 1.117,
  "prefill_ms_fp16": 34.8,
  "peak_vram_gb_fp16": 3.1285,
  "prefill_ms_nf4": 67.0601,
  "peak_vram_gb_nf4": 1.1993,
  "prefill_ms_fp4": 56.7417,
  "peak_vram_gb_fp4": 1.2583,
  "prefill_ms_int8": 178.8601,
  "peak_vram_gb_int8": 1.8389,
  "overhead_vs_prefill_pct": 3.2097
}


## 17. Figures — 300 dpi PNG + vector PDF

In [74]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 9, "font.family": "serif", "axes.grid": True,
                     "grid.alpha": .3, "figure.dpi": 120, "savefig.bbox": "tight"})
FIGW = 3.4   # IEEE single column, inches

def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(DIRS["figs"], f"{name}.{ext}"), dpi=300)
    plt.close(fig); print("fig:", name)

# --- F1 degradation rate by task x precision, with noise floor
fig, ax = plt.subplots(figsize=(FIGW, 2.4))
piv = (LAB[LAB.model == PRIMARY].groupby(["task", "precision"]).y.mean().unstack())
cols = [c for c in PRECISIONS if c in piv.columns]
piv[cols].plot.bar(ax=ax, width=.75, edgecolor="k", linewidth=.4)
if "fp16_selfrep" in piv.columns:
    for i, t in enumerate(piv.index):
        ax.hlines(piv.loc[t, "fp16_selfrep"], i - .4, i + .4, color="k", ls="--", lw=1.1)
ax.set_ylabel("degradation rate"); ax.set_xlabel("")
ax.legend(title=None, fontsize=6, ncol=2)
ax.set_title("Degradation vs. fp16 (dashed = fp16 noise floor)", fontsize=8)
savefig(fig, "F1_degradation_rates")

# --- F2 main AUC comparison with CIs
for prec in [p for p in CV if p != "fp16_selfrep"]:
    R = CV[prec]["R"]
    fig, ax = plt.subplots(figsize=(FIGW, 2.3))
    ax.barh(R.arm, R.auc, xerr=R.auc_ci, color="#4C72B0", edgecolor="k", linewidth=.4)
    ax.axvline(.5, color="k", ls=":", lw=1)
    tid = R[R.arm == "task_id_only"].auc.iloc[0]
    ax.axvline(tid, color="crimson", ls="--", lw=1, label="task-ID control")
    ax.set_xlim(.4, 1.0); ax.set_xlabel("ROC-AUC (mean $\\pm$ 95% CI)")
    ax.legend(fontsize=6); ax.set_title(f"{prec}", fontsize=8)
    savefig(fig, f"F2_auc_arms_{prec}")

# --- F3 ROC + PR curves
for prec in [p for p in CV if p != "fp16_selfrep"]:
    D, oof = DESIGNS[prec], CV[prec]["oof"]
    fig, axes = plt.subplots(1, 2, figsize=(2 * FIGW, 2.4))
    for arm in ["task_id_only", "length_rule", "tier0", "tier0+tier1"]:
        if arm not in oof: continue
        fpr, tpr, _ = roc_curve(D.y, oof[arm])
        axes[0].plot(fpr, tpr, lw=1.2,
                     label=f"{arm} ({roc_auc_score(D.y, oof[arm]):.3f})")
        pr, rc, _ = precision_recall_curve(D.y, oof[arm])
        axes[1].plot(rc, pr, lw=1.2, label=arm)
    axes[0].plot([0, 1], [0, 1], "k:", lw=1)
    axes[1].axhline(D.y.mean(), color="k", ls=":", lw=1)
    axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].legend(fontsize=6)
    axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision"); axes[1].legend(fontsize=6)
    fig.suptitle(f"{prec}", fontsize=8)
    savefig(fig, f"F3_roc_pr_{prec}")

# --- F4 within-task AUC heatmap (the control figure)
for prec in WT:
    if prec == "fp16_selfrep": continue
    cols = [c for c in WT[prec].columns if c not in ("arm", "macro")]
    W = WT[prec].set_index("arm")[cols]
    fig, ax = plt.subplots(figsize=(FIGW, 2.2))
    im = ax.imshow(W.values, cmap="RdYlBu", vmin=.4, vmax=.9, aspect="auto")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, fontsize=6)
    ax.set_yticks(range(len(W))); ax.set_yticklabels(W.index, fontsize=6)
    for i in range(W.shape[0]):
        for j in range(W.shape[1]):
            v = W.values[i, j]
            if np.isfinite(v): ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=5.5)
    ax.grid(False); fig.colorbar(im, ax=ax, label="within-task AUC")
    ax.set_title(f"{prec}: AUC inside each task", fontsize=8)
    savefig(fig, f"F4_within_task_{prec}")

# --- F5 leave-one-task-out
for prec in LT:
    if prec == "fp16_selfrep": continue
    cols = [c for c in LT[prec].columns if c not in ("arm", "macro")]
    L = LT[prec].set_index("arm")[cols]
    fig, ax = plt.subplots(figsize=(FIGW, 2.1))
    L.plot.bar(ax=ax, width=.75, edgecolor="k", linewidth=.4)
    ax.axhline(.5, color="k", ls=":", lw=1); ax.set_ylabel("AUC on held-out task")
    ax.set_xlabel(""); ax.legend(fontsize=6, title=None)
    ax.set_title(f"{prec}: leave-one-task-out", fontsize=8)
    savefig(fig, f"F5_loto_{prec}")

# --- F6 feature ablation
for prec, A in ABL.items():
    fig, ax = plt.subplots(figsize=(FIGW, 2.1))
    a = A[A.removed != "none (full)"].sort_values("delta")
    ax.barh(a.removed, a.delta, color="#DD8452", edgecolor="k", linewidth=.4)
    ax.axvline(0, color="k", lw=1)
    ax.set_xlabel("$\\Delta$ AUC when group removed")
    ax.set_title(f"{prec}", fontsize=8)
    savefig(fig, f"F6_ablation_{prec}")

# --- F7 feature importance
for prec, s in IMP.items():
    fig, ax = plt.subplots(figsize=(FIGW, 2.6))
    s.head(15)[::-1].plot.barh(ax=ax, color="#55A868", edgecolor="k", linewidth=.4)
    ax.set_xlabel("gain importance"); ax.tick_params(axis="y", labelsize=6)
    ax.set_title(f"{prec}: top features", fontsize=8)
    savefig(fig, f"F7_importance_{prec}")

# --- F8 ROUTER (headline)
for prec in set(p for p, _ in ROUTE):
    fig, ax = plt.subplots(figsize=(FIGW, 2.4))
    c = ROUTE[(prec, "tier0")]
    ax.plot(c.coverage, c.oracle, "k--", lw=1, label="oracle")
    ax.plot(c.coverage, c.model, lw=1.6, label="tier-0 router")
    if (prec, "tier0+tier1") in ROUTE:
        ax.plot(ROUTE[(prec, "tier0+tier1")].coverage,
                ROUTE[(prec, "tier0+tier1")].model, lw=1.6, label="tier-0+1 router")
    ax.plot(c.coverage, c["random"], ":", lw=1.2, label="random")
    ax.axhline(c.always_quant.iloc[0], color="grey", lw=1, label=f"always {prec}")
    ax.set_xlabel("fraction escalated to fp16"); ax.set_ylabel("fp16 parity retained")
    ax.legend(fontsize=6, loc="lower right"); ax.set_title(f"{prec}", fontsize=8)
    savefig(fig, f"F8_router_{prec}")


# --- F10 calibration
for prec in [p for p in CV if p != "fp16_selfrep"]:
    D, s = DESIGNS[prec], CV[prec]["oof"]["tier0+tier1"]
    q = pd.qcut(s, 10, duplicates="drop")
    g = pd.DataFrame({"s": s, "y": D.y.values, "q": q}).groupby("q", observed=True).mean(
        numeric_only=True)
    fig, ax = plt.subplots(figsize=(FIGW, 2.2))
    ax.plot([0, 1], [0, 1], "k:", lw=1); ax.plot(g.s, g.y, "o-", lw=1.2, ms=3)
    ax.set_xlabel("predicted"); ax.set_ylabel("observed"); ax.set_title(f"{prec}", fontsize=8)
    savefig(fig, f"F10_calibration_{prec}")

fig: F1_degradation_rates
fig: F2_auc_arms_nf4
fig: F2_auc_arms_fp4
fig: F2_auc_arms_int8
fig: F3_roc_pr_nf4
fig: F3_roc_pr_fp4
fig: F3_roc_pr_int8
fig: F4_within_task_nf4
fig: F4_within_task_fp4
fig: F4_within_task_int8
fig: F5_loto_nf4
fig: F5_loto_fp4
fig: F5_loto_int8
fig: F6_ablation_nf4
fig: F6_ablation_fp4
fig: F6_ablation_int8
fig: F7_importance_nf4
fig: F7_importance_fp4
fig: F7_importance_int8
fig: F8_router_int8
fig: F8_router_nf4
fig: F8_router_fp4
fig: F10_calibration_nf4
fig: F10_calibration_fp4
fig: F10_calibration_int8


In [75]:
print("TASKS config :", TASKS)
print("in PROMPTS   :", sorted(PROMPTS.task.unique()))
print("in DF        :", sorted(DF.task.unique()))
print("in LAB       :", sorted(LAB.task.unique()))
print("rows/task    :\n", DF[DF.run_tag=="a"].groupby(["task","precision"]).size().unstack())

TASKS config : ['gsm8k', 'gsm8k_direct', 'arc_easy', 'arc_challenge', 'sst2']
in PROMPTS   : ['arc_challenge', 'arc_easy', 'gsm8k', 'gsm8k_direct', 'sst2']
in DF        : ['arc_challenge', 'arc_easy', 'gsm8k', 'gsm8k_direct', 'sst2']
in LAB       : ['arc_challenge', 'arc_easy', 'gsm8k', 'gsm8k_direct', 'sst2']
rows/task    :
 precision      fp16   fp4  int8   nf4
task                                 
arc_challenge  1000  1000  1000  1000
arc_easy       1500  1500  1500  1500
gsm8k          1000  1000  1000  1000
gsm8k_direct   1000  1000  1000  1000
sst2            872   872   872   872


## 17b. Dataset profile, learning curves, confusion matrices, threshold metrics

Everything a reviewer expects to see but that AUC alone does not show. Note there is no neural
training loop here, so the analogue of a training curve is **boosting-round** train-vs-validation
loss (overfitting evidence) plus a **sample-size** learning curve (does more data help?).

In [76]:
from sklearn.metrics import (confusion_matrix, precision_recall_fscore_support,
                             matthews_corrcoef, balanced_accuracy_score, accuracy_score)
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split

# ---------------------------------------------------------------- F11 dataset profile
fig, axes = plt.subplots(1, 3, figsize=(3 * FIGW, 2.2))
tl = T0.merge(PROMPTS[["uid", "task"]], on="uid")
for t in TASKS:
    axes[0].hist(tl[tl.task == t].t0_n_tokens, bins=30, alpha=.55, label=t)
axes[0].set_xlabel("prompt length (tokens)"); axes[0].set_ylabel("count")
axes[0].legend(fontsize=6); axes[0].set_title("Prompt length", fontsize=8)

for t in TASKS:
    axes[1].hist(tl[tl.task == t].t0_frag_ratio, bins=30, alpha=.55, label=t)
axes[1].set_xlabel("subwords per word"); axes[1].set_title("Fragmentation", fontsize=8)

bal = LAB[LAB.model == PRIMARY].groupby(["task", "precision"]).y.mean().unstack()
bal[[c for c in PRECISIONS if c in bal.columns]].plot.bar(ax=axes[2], width=.75,
                                                          edgecolor="k", linewidth=.4)
axes[2].set_ylabel("positive-class rate"); axes[2].set_xlabel("")
axes[2].legend(fontsize=6); axes[2].set_title("Class balance", fontsize=8)
savefig(fig, "F11_dataset_profile")

# ---------------------------------------------------------------- F12 boosting curves
from xgboost import XGBClassifier
CURVES = {}
for prec in [p for p in CV if p != "fp16_selfrep"]:
    D = DESIGNS[prec]; X = D[F_T0 + F_T1].values; y = D.y.values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, stratify=y,
                                          random_state=SEED)
    spw = float((len(ytr) - ytr.sum()) / max(ytr.sum(), 1))
    c = XGBClassifier(n_estimators=400, max_depth=4, learning_rate=.07, subsample=.85,
                      colsample_bytree=.85, reg_lambda=1.5, n_jobs=4, random_state=SEED,
                      scale_pos_weight=spw, eval_metric=["logloss", "auc"])
    c.fit(Xtr, ytr, eval_set=[(Xtr, ytr), (Xte, yte)], verbose=False)
    r = c.evals_result(); CURVES[prec] = r
    fig, axes = plt.subplots(1, 2, figsize=(2 * FIGW, 2.2))
    for ax, met, lab in zip(axes, ["logloss", "auc"], ["log loss", "ROC-AUC"]):
        ax.plot(r["validation_0"][met], lw=1.3, label="train")
        ax.plot(r["validation_1"][met], lw=1.3, label="validation")
        ax.set_xlabel("boosting round"); ax.set_ylabel(lab); ax.legend(fontsize=6)
    fig.suptitle(f"{prec}: training vs validation", fontsize=8)
    savefig(fig, f"F12_training_curve_{prec}")

# ---------------------------------------------------------------- F13 sample-size curve
LCURVE = {}
for prec in [p for p in CV if p != "fp16_selfrep"]:
    D = DESIGNS[prec]; X = D[F_T0].values; y = D.y.values
    fracs = np.linspace(.15, 1.0, 8); rows = []
    for f in fracs:
        sc = []
        for rep in range(5):
            sss = StratifiedShuffleSplit(n_splits=1, test_size=.25, random_state=rep)
            tr, te = next(sss.split(X, y))
            k = max(int(f * len(tr)), 30)
            sub = np.random.RandomState(rep).choice(tr, k, replace=False)
            if y[sub].sum() < 5: continue
            m = xgb(y[sub]); m.fit(X[sub], y[sub])
            sc.append(roc_auc_score(y[te], m.predict_proba(X[te])[:, 1]))
        if sc: rows.append(dict(frac=f, n=int(f * .75 * len(y)),
                                auc=np.mean(sc), sd=np.std(sc, ddof=1) if len(sc) > 1 else 0))
    LCURVE[prec] = pd.DataFrame(rows)
    L = LCURVE[prec]
    fig, ax = plt.subplots(figsize=(FIGW, 2.1))
    ax.errorbar(L.n, L.auc, yerr=L.sd, marker="o", ms=3, lw=1.3, capsize=2)
    ax.axhline(.5, color="k", ls=":", lw=1)
    ax.set_xlabel("training examples"); ax.set_ylabel("held-out ROC-AUC")
    ax.set_title(f"{prec}: sample-size learning curve", fontsize=8)
    savefig(fig, f"F13_learning_curve_{prec}")

fig: F11_dataset_profile
fig: F12_training_curve_nf4
fig: F12_training_curve_fp4
fig: F12_training_curve_int8
fig: F13_learning_curve_nf4
fig: F13_learning_curve_fp4
fig: F13_learning_curve_int8


In [77]:
# ---------------------------------------------------------------- F14/F15 thresholds + CM
def thr_metrics(y, s, thr):
    yh = (s >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    p, r, f1, _ = precision_recall_fscore_support(y, yh, average="binary",
                                                  zero_division=0)
    return dict(threshold=thr, accuracy=accuracy_score(y, yh),
                balanced_acc=balanced_accuracy_score(y, yh),
                precision=p, recall=r, f1=f1, specificity=tn / max(tn + fp, 1),
                mcc=matthews_corrcoef(y, yh) if len(set(yh)) > 1 else 0.0,
                TP=tp, FP=fp, TN=tn, FN=fn)

METRICS, CMS = [], {}
for prec in [p for p in CV if p != "fp16_selfrep"]:
    D = DESIGNS[prec]; y = D.y.values
    for arm in ["tier0", "tier0+tier1"]:
        s = CV[prec]["oof"][arm]
        grid = np.linspace(.02, .98, 97)
        sweep_ = pd.DataFrame([thr_metrics(y, s, t) for t in grid])
        t_f1 = float(sweep_.loc[sweep_.f1.idxmax(), "threshold"])
        t_cov = float(np.quantile(s, 1 - 0.15))          # 15% escalation budget
        for nm, t in [("default_0.5", .5), ("f1_optimal", t_f1), ("cov15", t_cov)]:
            METRICS.append(dict(quant=prec, arm=arm, operating_point=nm,
                                **thr_metrics(y, s, t)))   # 'quant' avoids the
            # name clash with the 'precision' metric
        CMS[(prec, arm)] = (sweep_, t_f1, t_cov, y, s)

    # threshold sweep figure (tier0 only, to keep the figure readable)
    sweep_, t_f1, t_cov, y_, s_ = CMS[(prec, "tier0")]
    fig, ax = plt.subplots(figsize=(FIGW, 2.1))
    for col in ["precision", "recall", "f1"]:
        ax.plot(sweep_.threshold, sweep_[col], lw=1.3, label=col)
    ax.axvline(t_f1, color="crimson", ls="--", lw=1, label="F1-optimal")
    ax.axvline(t_cov, color="grey", ls=":", lw=1, label="15% budget")
    ax.set_xlabel("decision threshold"); ax.legend(fontsize=6)
    ax.set_title(f"{prec}: tier-0", fontsize=8)
    savefig(fig, f"F15_threshold_sweep_{prec}")

    # confusion matrices at the two meaningful operating points
    fig, axes = plt.subplots(1, 2, figsize=(2 * FIGW, 2.0))
    for ax, (nm, t) in zip(axes, [("F1-optimal", t_f1), ("15% escalation budget", t_cov)]):
        cm = confusion_matrix(y_, (s_ >= t).astype(int), labels=[0, 1])
        ax.imshow(cm, cmap="Blues"); ax.grid(False)
        ax.set_xticks([0, 1]); ax.set_xticklabels(["pred 0", "pred 1"], fontsize=7)
        ax.set_yticks([0, 1]); ax.set_yticklabels(["true 0", "true 1"], fontsize=7)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{cm[i,j]}\n({cm[i,j]/cm.sum()*100:.1f}%)",
                        ha="center", va="center", fontsize=7,
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
        ax.set_title(f"{nm} (t={t:.2f})", fontsize=7.5)
    fig.suptitle(f"{prec}: tier-0 confusion matrices", fontsize=8)
    savefig(fig, f"F14_confusion_{prec}")

METRICS = pd.DataFrame(METRICS)
print(METRICS.round(3).to_string(index=False))

fig: F15_threshold_sweep_nf4
fig: F14_confusion_nf4
fig: F15_threshold_sweep_fp4
fig: F14_confusion_fp4
fig: F15_threshold_sweep_int8
fig: F14_confusion_int8
quant         arm operating_point  threshold  accuracy  balanced_acc  precision  recall    f1  specificity   mcc  TP  FP   TN  FN
  nf4       tier0     default_0.5      0.500     0.819         0.597      0.269   0.307 0.287        0.888 0.184 132 359 2836 298
  nf4       tier0      f1_optimal      0.370     0.769         0.640      0.249   0.470 0.326        0.810 0.217 202 608 2587 228
  nf4       tier0           cov15      0.476     0.811         0.606      0.267   0.337 0.298        0.875 0.192 145 399 2796 285
  nf4 tier0+tier1     default_0.5      0.500     0.857         0.679      0.407   0.444 0.425        0.913 0.344 191 278 2917 239
  nf4 tier0+tier1      f1_optimal      0.290     0.817         0.740      0.350   0.640 0.453        0.840 0.377 275 510 2685 155
  nf4 tier0+tier1           cov15      0.441     0.847        

## 18. Cross-precision transfer (no extra generation)

Train the predictor on one quantization tier, test it on another. This needs no new inference:
same prompts, same features, labels you already have.

It answers a sharper question than cross-model transfer does. If a predictor trained on NF4
degradation ranks FP4 degradation well, then fragility is a property of the **input**, not of
the particular quantization scheme -- which is the generalization claim the paper actually
wants to make.

Read the off-diagonal against the diagonal. Off-diagonal close to diagonal means fragility
transfers. Off-diagonal near 0.5 means each scheme breaks on different prompts, and the claim
must be narrowed to per-scheme prediction. Diagonal cells use out-of-fold scores, never
in-sample fits.

In [78]:
XPREC = None
_qp = [p for p in CV if p != "fp16_selfrep"]
if len(_qp) > 1:
    rows = []
    for a in _qp:
        for b in _qp:
            Da, Db = DESIGNS[a], DESIGNS[b]
            if Da.y.sum() < 20 or Db.y.sum() < 20: continue
            arms = [("tier0", F_T0)] + ([("tier0+tier1", F_T0 + F_T1)] if F_T1 else [])
            for arm, feats in arms:
                if a == b:
                    auc = roc_auc_score(Db.y.values, CV[b]["oof"][arm])
                else:
                    c = xgb(Da.y.values); c.fit(Da[feats].values, Da.y.values)
                    auc = roc_auc_score(Db.y.values,
                                        c.predict_proba(Db[feats].values)[:, 1])
                rows.append(dict(train=a, test=b, arm=arm, auc=auc, same=(a == b)))
    XPREC = pd.DataFrame(rows)
    print(XPREC.round(3).to_string(index=False))

    piv = XPREC[XPREC.arm == "tier0"].pivot(index="train", columns="test", values="auc")
    fig, ax = plt.subplots(figsize=(FIGW, 2.1))
    im = ax.imshow(piv.values, cmap="RdYlBu", vmin=.5, vmax=.9)
    ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns, fontsize=7)
    ax.set_yticks(range(len(piv.index)));  ax.set_yticklabels(piv.index, fontsize=7)
    ax.set_xlabel("tested on"); ax.set_ylabel("trained on")
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.values[i, j]
            if np.isfinite(v): ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=7)
    ax.grid(False); fig.colorbar(im, ax=ax, label="ROC-AUC")
    ax.set_title("Cross-precision transfer (tier-0)", fontsize=8)
    savefig(fig, "F16_cross_precision")
else:
    print("Need >=2 quantized tiers with enough positives for cross-precision transfer.")

train test         arm   auc  same
  nf4  nf4       tier0 0.697  True
  nf4  nf4 tier0+tier1 0.851  True
  nf4  fp4       tier0 0.752 False
  nf4  fp4 tier0+tier1 0.897 False
  nf4 int8       tier0 0.843 False
  nf4 int8 tier0+tier1 0.868 False
  fp4  nf4       tier0 0.774 False
  fp4  nf4 tier0+tier1 0.869 False
  fp4  fp4       tier0 0.737  True
  fp4  fp4 tier0+tier1 0.920  True
  fp4 int8       tier0 0.833 False
  fp4 int8 tier0+tier1 0.843 False
 int8  nf4       tier0 0.695 False
 int8  nf4 tier0+tier1 0.807 False
 int8  fp4       tier0 0.704 False
 int8  fp4 tier0+tier1 0.789 False
 int8 int8       tier0 0.729  True
 int8 int8 tier0+tier1 0.809  True
fig: F16_cross_precision


## 19. LaTeX tables (IEEEtran-ready)

In [79]:
def to_tex(df, name, caption, label, fmt="%.3f", idx=False):
    body = df.to_latex(index=idx, float_format=lambda x: fmt % x, escape=True,
                       column_format="l" + "r" * (df.shape[1] - (0 if idx else 1)))
    tex = ("\\begin{table}[t]\n\\centering\n\\caption{" + caption + "}\n\\label{" + label +
           "}\n\\resizebox{\\columnwidth}{!}{%\n" + body + "}\n\\end{table}\n")
    open(os.path.join(DIRS["tables"], name + ".tex"), "w").write(tex)
    print("table:", name)

# T1 corpus composition and fp16 accuracy
acc = (DF[(DF.model == PRIMARY) & (DF.run_tag == "a")]
       .groupby(["task", "precision"]).correct.mean().unstack().reset_index())
to_tex(acc, "T1_accuracy", "Task accuracy by numerical precision.", "tab:acc")

# T2 degradation + noise floor
dg = LAB[LAB.model == PRIMARY].groupby(["task", "precision"]).y.agg(["mean", "sum", "size"])
to_tex(dg.reset_index(), "T2_degradation",
       "Degradation rate on the fp16-correct subset. The \\texttt{fp16\\_selfrep} row is the "
       "run-to-run noise floor.", "tab:degrade")

# T2b flip decomposition
to_tex(FLIP, "T2b_flip_decomposition",
       "Flip decomposition. Aggregate accuracy nets degradations against recoveries; "
       "\\emph{churn} counts every changed prediction.", "tab:flip")

# T2c length-vs-content contrast
if len(CONTRAST):
    to_tex(CONTRAST, "T2c_length_vs_content",
           "Controlled contrasts. \\texttt{gsm8k}/\\texttt{gsm8k\\_direct} share items and "
           "differ only in generated length.", "tab:contrast")

# T3 main results
for prec in [p for p in CV if p != "fp16_selfrep"]:
    R = CV[prec]["R"][["arm", "auc", "auc_ci", "ap", "brier"]]
    to_tex(R, f"T3_main_{prec}",
           f"Meta-classifier performance for {prec} "
           f"(mean over {N_SPLITS}$\\times${N_REPEATS} stratified CV, 95\\% CI).",
           f"tab:main_{prec}")

# T4 controls
for prec in WT:
    if prec == "fp16_selfrep": continue
    to_tex(WT[prec], f"T4_within_{prec}",
           f"Within-task AUC ({prec}). Confound control.", f"tab:within_{prec}")
    to_tex(LT[prec], f"T4_loto_{prec}",
           f"Leave-one-task-out AUC ({prec}).", f"tab:loto_{prec}")

# T5 ablation
for prec, A in ABL.items():
    to_tex(A, f"T5_ablation_{prec}", f"Feature-group ablation ({prec}).", f"tab:abl_{prec}")

# T12 paired arm comparison
for prec, c in CMP.items():
    to_tex(c[["arm", "auc_arm", "delta", "p_wilcoxon", "verdict"]],
           f"T12_arm_comparison_{prec}",
           f"Paired comparison of classifier arms on identical folds ({prec}), "
           f"Wilcoxon signed-rank.", f"tab:arms_{prec}")

# T6 router operating points
to_tex(OPS, "T6_router",
       "Router operating points. \\emph{parity} is the fraction of fp16 accuracy retained; "
       "\\emph{lost\\_recovered} is the share of quantization-induced loss recovered.",
       "tab:router")

# T9 threshold-level classification metrics
to_tex(METRICS, "T9_threshold_metrics",
       "Classification metrics at three operating points, computed from out-of-fold scores. "
       "\\emph{cov15} is the threshold that escalates 15\\% of prompts.",
       "tab:threshold")

# T10 sample-size learning curve
for prec, L in LCURVE.items():
    to_tex(L, f"T10_learning_curve_{prec}",
           f"Held-out AUC against training-set size ({prec}).", f"tab:lcurve_{prec}")

# T7 cost
to_tex(pd.DataFrame([COST]).T.reset_index().rename(columns={"index": "metric", 0: "value"}),
       "T7_cost", "Router overhead and per-precision inference cost.", "tab:cost", fmt="%.3f")

if XPREC is not None and len(XPREC):
    to_tex(XPREC, "T11_cross_precision",
           "Cross-precision transfer. Diagonal entries are out-of-fold; off-diagonal "
           "entries train on one quantization tier and test on another.",
           "tab:xprec")

table: T1_accuracy
table: T2_degradation
table: T2b_flip_decomposition
table: T2c_length_vs_content
table: T3_main_nf4
table: T3_main_fp4
table: T3_main_int8
table: T4_within_nf4
table: T4_loto_nf4
table: T4_within_fp4
table: T4_loto_fp4
table: T4_within_int8
table: T4_loto_int8
table: T5_ablation_nf4
table: T5_ablation_fp4
table: T5_ablation_int8
table: T12_arm_comparison_nf4
table: T12_arm_comparison_fp4
table: T12_arm_comparison_int8
table: T6_router
table: T9_threshold_metrics
table: T10_learning_curve_nf4
table: T10_learning_curve_fp4
table: T10_learning_curve_int8
table: T7_cost
table: T11_cross_precision


## 20. Paper-ready summary

In [80]:
S = {"config": dict(models=RUN_MODELS, precisions=PRECISIONS, tasks=TASKS,
                    n_prompts=int(len(PROMPTS)), cv=f"{N_SPLITS}x{N_REPEATS}", smoke=SMOKE),
     "cost": COST, "gate": G.to_dict("records")}
lines = []
for prec in [p for p in CV if p != "fp16_selfrep"]:
    R, D = CV[prec]["R"], DESIGNS[prec]
    best = R.iloc[0]; tid = R[R.arm == "task_id_only"].iloc[0]
    lr = R[R.arm == "length_rule"].iloc[0]
    t0 = R[R.arm == "tier0"].iloc[0]
    wt = WT[prec][WT[prec].arm == "tier0"].macro.iloc[0]
    lt = LT[prec][LT[prec].arm == "tier0"].macro.iloc[0]
    op = OPS[(OPS.precision == prec) & (OPS.arm == "tier0") & (OPS.coverage == 0.15)]
    S[prec] = dict(pos_rate=float(D.y.mean()), n=int(len(D)),
                   best_arm=best.arm, best_auc=float(best.auc), best_ci=float(best.auc_ci),
                   tier0_auc=float(t0.auc), taskid_auc=float(tid.auc),
                   length_rule_auc=float(lr.auc), within_task_macro=float(wt),
                   loto_macro=float(lt), p_vs_taskid=float(R.attrs["p"]),
                   parity_at_15pct=float(op.parity.iloc[0]) if len(op) else None,
                   recovered_at_15pct=float(op.lost_recovered.iloc[0]) if len(op) else None)
    lines += [
      f"[{prec}] {D.y.mean()*100:.1f}% of fp16-correct prompts degrade (n={len(D)}).",
      f"  Tier-0 AUC {t0.auc:.3f} +/- {t0.auc_ci:.3f} vs task-ID control {tid.auc:.3f} "
      f"and length-rule {lr.auc:.3f} (p={R.attrs['p']:.1e}).",
      f"  Within-task macro AUC {wt:.3f}; leave-one-task-out {lt:.3f}.",
      (f"  Escalating 15% of prompts retains {op.parity.iloc[0]*100:.1f}% of fp16 accuracy, "
       f"recovering {op.lost_recovered.iloc[0]*100:.0f}% of the loss." if len(op) else ""),
    ]
if "fp16_selfrep" in CV:
    S["negative_control_auc"] = float(CV["fp16_selfrep"]["R"].auc.max())
    lines.append(f"[control] best AUC predicting fp16-vs-fp16 nondeterminism = "
                 f"{S['negative_control_auc']:.3f} (should be near 0.5).")
elif DESIGNS.get("fp16_selfrep") is not None:
    _d = DESIGNS["fp16_selfrep"]
    S["negative_control_rate"] = float(_d.y.mean())
    S["negative_control_n_pos"] = int(_d.y.sum())
    lines.append(f"[control] fp16-vs-fp16 disagreement {int(_d.y.sum())}/{len(_d)} "
                 f"({_d.y.mean()*100:.2f}%); too few positives to model, i.e. decoding was "
                 f"effectively deterministic.")
lines.append(f"[cost] router overhead {COST['router_total_ms']:.3f} ms/prompt.")

S["pooled"] = POOLED
if CONS: S["consistent_fragility"] = CONS
lines.append(f"[pooled] AUC {POOLED['auc']:.3f} +/- {POOLED['ci']:.3f} "
             f"over {POOLED['pos']} positives.")
if CONS:
    lines.append(f"[consistent] AUC {CONS['auc']:.3f} +/- {CONS['ci']:.3f} on the "
                 f"{CONS['kept_frac']*100:.0f}% of prompts with unambiguous labels.")
json.dump(S, open(os.path.join(DIRS["results"], "summary.json"), "w"), indent=2, default=str)
print("\n".join([l for l in lines if l]))
print("\nfigures ->", DIRS["figs"], "\ntables  ->", DIRS["tables"])

[nf4] 11.9% of fp16-correct prompts degrade (n=3625).
  Tier-0 AUC 0.695 +/- 0.008 vs task-ID control 0.722 and length-rule 0.699 (p=1.2e-03).
  Within-task macro AUC 0.536; leave-one-task-out 0.531.
  Escalating 15% of prompts retains 91.9% of fp16 accuracy, recovering 32% of the loss.
[fp4] 17.2% of fp16-correct prompts degrade (n=3625).
  Tier-0 AUC 0.741 +/- 0.010 vs task-ID control 0.760 and length-rule 0.673 (p=2.5e-04).
  Within-task macro AUC 0.505; leave-one-task-out 0.475.
  Escalating 15% of prompts retains 89.6% of fp16 accuracy, recovering 40% of the loss.
[int8] 3.6% of fp16-correct prompts degrade (n=3625).
  Tier-0 AUC 0.724 +/- 0.018 vs task-ID control 0.763 and length-rule 0.686 (p=3.1e-02).
  Within-task macro AUC 0.552; leave-one-task-out 0.537.
  Escalating 15% of prompts retains 98.0% of fp16 accuracy, recovering 45% of the loss.
[control] fp16-vs-fp16 disagreement 9/3625 (0.25%); too few positives to model, i.e. decoding was effectively deterministic.
[cost] rout

## 21. Resuming after a dropped session

**Resuming after a dropped session:** re-run §0, §0b, §1–4 (definitions only, seconds), call
`resume_status()`, then re-run §5/§7. Anything already on disk is skipped. Model weights come
from the Drive-backed `HF_HOME`, so no re-download.